# 梯度下降法(Gradient Descent，简称GD)

梯度下降法是机器学习，特别是神经网络有指导学习(深度学习)训练算法的基础。梯度下降法的核心逻辑是：对于参数可导的损失函数（允许有限个不可导点），负梯度方向是函数值在局部下降最快的方向。通过沿负梯度方向迭代更新参数，算法可以收敛到起点附近的驻点解(可能是最优解)。

> **核心定位**：梯度下降法（GD）是所有一阶优化算法的**数学基础**与**理论原点**。理解梯度下降法，就是理解所有优化器的起点。

## 先睹为快：一个典型的梯度下降算例

让我们从一个简单的**二元对称凸二次函数**出发，通过代码直观地观察梯度下降法的工作过程。为什么选择二元函数？因为一元函数的梯度退化为标量导数，无法展现梯度作为**向量**的核心特征——方向与大小。

**目标函数**：
$$
f(x, y) = x^2 + y^2
$$

这是一个开口向上的椭圆抛物面，x方向和y方向的曲率相同（对称），其**理论最优解**为 $(x^*, y^*) = (0, 0)$（令 $\nabla f = (2x, 2y)^\top = \mathbf{0}$ 得到）。

**梯度向量**：
$$
\nabla f(x, y) = \begin{bmatrix} \frac{\partial f}{\partial x} \\ \frac{\partial f}{\partial y} \end{bmatrix} = \begin{bmatrix} 2x \\ 2y \end{bmatrix}
$$

**梯度下降更新规则**：
$$
\begin{bmatrix} x_{k+1} \\ y_{k+1} \end{bmatrix} = \begin{bmatrix} x_k \\ y_k \end{bmatrix} - \eta \cdot \nabla f(x_k, y_k)
$$

其中 $\eta$ 是步长。

In [ ]:
# ===== 导入必要的库 =====
import numpy as np
import plotly.graph_objects as go

# ========= 参数设置 =========
x0, y0 = 2.5, 2.5
eta = 0.1
max_iter = 40

# ========= 定义二元目标函数及其梯度 =========
def f(x, y):
    return x**2 + y**2

def grad_f(x, y):
    return np.array([2*x, 2*y])


# ========= 运行梯度下降 =========
points = [(x0, y0)]
values = [f(x0, y0)]
x_cur, y_cur = x0, y0

print("【梯度下降法迭代过程】η = 0.1，从 (x₀, y₀) = (2.5, 2.5) 出发")
print("=" * 140)

# 表头及对齐工具
headers = ["步数", "当前位置(x,y)", "梯度∇f", "更新量-η·∇f", "下一步位置", "函数值下降量"]
w_k, w_xy1, w_grad, w_step, w_next, w_dec = 6, 24, 24, 26, 24, 16

def fnt(text, width, align='l'):
    text = str(text)
    padding = width - len(text)
    if padding < 0:
        return text
    if align == 'l':
        return text + ' ' * padding
    return ' ' * padding + text

header_str = (fnt(headers[0], w_k, 'l') + fnt(headers[1], w_xy1, 'l') + 
              fnt(headers[2], w_grad, 'l') + fnt(headers[3], w_step, 'l') + 
              fnt(headers[4], w_next, 'l') + fnt(headers[5], w_dec, 'l'))
print(header_str)
print("=" * 140)

# ===== 梯度下降主循环 =====
for k in range(max_iter):
    g = grad_f(x_cur, y_cur)
    step = -eta * g
    x_next = x_cur + step[0]
    y_next = y_cur + step[1]
    f_cur = f(x_cur, y_cur)
    f_next = f(x_next, y_next)
    decrease = f_cur - f_next
    
    # 记录路径点
    points.append((x_next, y_next))
    values.append(f_next)
    
    # 打印部分迭代过程
    if k < 20 or k >= max_iter - 5:
        cur_str = f"({x_cur:>8.4f}, {y_cur:>8.4f})"
        grad_str = f"({g[0]:>8.4f}, {g[1]:>8.4f})"
        step_str = f"({step[0]:>9.4f}, {step[1]:>9.4f})"
        next_str = f"({x_next:>8.4f}, {y_next:>8.4f})"
        dec_str = f"{decrease:>12.6f}"

        row_str = (fnt(k, w_k, 'l') + fnt(cur_str, w_xy1, 'l') + 
                   fnt(grad_str, w_grad, 'l') + fnt(step_str, w_step, 'l') + 
                   fnt(next_str, w_next, 'l') + fnt(dec_str, w_dec, 'r'))
        print(row_str)
    elif k == 20:
        print("... (中间迭代省略) ...")
    
    x_cur, y_cur = x_next, y_next

print("=" * 140)

# ================= 补充最后的控制台输出（包含起点） =================
print(f"【理论最优解】 (x*, y*) = (0.0000, 0.0000), f(x*, y*) = 0.0000")
print(f"【起始点】  (x₀, y₀) = ({points[0][0]:.6f}, {points[0][1]:.6f}), f(x₀, y₀) = {values[0]:.6f}")
print(f"【迭代终点】 迭代{max_iter}步后: (x, y) = ({points[-1][0]:.6f}, {points[-1][1]:.6f}), f = {values[-1]:.6f}")
print(f"【总下降量】  {values[0] - values[-1]:.6f}")

# 提取迭代路径的 X 和 Y 坐标
xs = np.array([p[0] for p in points])
ys = np.array([p[1] for p in points])

# ============================================================
# 图1：等高线图 + 梯度下降迭代路径（去掉多余刻度，保持1:1等比例）
# ============================================================
x_range = np.linspace(-4.0, 4.0, 400)
y_range = np.linspace(-3.0, 3.0, 400)
X, Y = np.meshgrid(x_range, y_range)
Z = f(X, Y)

fig1 = go.Figure()

# 1. 添加等高线
fig1.add_trace(go.Contour(
    x=x_range, y=y_range, z=Z,
    colorscale="Viridis",
    contours=dict(start=0.1, end=16, size=1.0, coloring="heatmap"),
    colorbar=dict(title="f(x,y)", thickness=15),
    showlegend=False
))

# 2. 标记极小值（五角星）
fig1.add_trace(go.Scatter(
    x=[0.0], y=[0.0], mode="markers",
    marker=dict(color="red", size=10, symbol="star"), name="极小值"
))

# 3. 绘制迭代路径
fig1.add_trace(go.Scatter(
    x=xs, y=ys, mode="lines+markers",
    line=dict(color="red", width=2),
    marker=dict(color="red", size=4),
    name="迭代路径"
))

# 4. 标记起点
fig1.add_trace(go.Scatter(
    x=[xs[0]], y=[ys[0]], mode="markers",
    marker=dict(color="red", size=8), name="起点"
))

# ===== 图1布局（紧凑、无多余刻度） =====
fig1.update_layout(
    title=dict(text=f"图1 迭代路径：f(x,y) = x² + y² (η={eta}, {max_iter}步)", font=dict(size=18), x=0.5),
    template="plotly_white",
    autosize=False,
    width=800, height=700,
    xaxis=dict(
        title="x",
        range=[-4.0, 4.0],
        dtick=1,
        showgrid=False,
        scaleanchor="y",
        scaleratio=1
    ),
    yaxis=dict(
        title="y",
        range=[-3.0, 3.0],
        dtick=1,
        showgrid=False,
        scaleanchor="x",
        scaleratio=1
    ),
    hovermode="x unified", 
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5, bgcolor="rgba(255,255,255,0.8)"),
    margin=dict(l=80, r=60, t=100, b=60)
)

fig1.update_xaxes(automargin=True)
fig1.update_yaxes(automargin=True)

fig1.show()

# ============================================================
# 图2：损失函数值下降曲线
# ============================================================
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=list(range(len(values))), y=values,
    mode="lines+markers",
    line=dict(color="royalblue", width=1.5),
    marker=dict(color="royalblue", size=3),
    name="损失值"
))

fig2.update_layout(
    title=dict(text=f"图2 损失函数值迭代变化：f(x,y) = x² + y² (η={eta}, {max_iter}步)", font=dict(size=18), x=0.5),
    xaxis_title="迭代步数 k", yaxis_title="f(x,y)",
    template="plotly_white",
    autosize=False,
    width=800, height=600,
    margin=dict(l=80, r=40, t=100, b=60),
    hovermode="x unified"
)

fig2.update_xaxes(automargin=True)
fig2.update_yaxes(automargin=True)

fig2.show()

**观察这个迭代过程**：

1. **梯度∇是一个向量**：在每个点 $(x,y)$，梯度 $\nabla f = (2x, 2y)^\top$ 既有**大小**（$\|\nabla f\| = \sqrt{4x^2 + 4y^2} = 2\sqrt{x^2+y^2}$）又有**方向**（指向函数值增长最快的方向）。负梯度 $-\nabla f$ 指向下降最快的方向。

2. **梯度方向垂直于等高线**：从等高线图上可以清晰看到，每一步的迭代方向（红色路径）都与等高线垂直——这是梯度下降法的几何本质：**沿等高线法线方向下山最快**。（该性质的严格数学证明详见附录A）

3. **对称曲率导致路径呈直线**：由于 $f(x,y) = x^2 + y^2$ 在 $x$ 方向和 $y$ 方向的曲率相同（系数均为1），梯度在两个方向的分量 $2x$ 和 $2y$ 具有相同的缩放比例。这导致迭代路径从起点笔直地指向最优解 $(0,0)$——这是对称凸函数的特征。

4. **梯度大小决定步幅**：初始点 $(2.5, 2.5)$ 处梯度 $\nabla f = (5, 5)^\top$，大小 $\|\nabla f\| = \sqrt{25 + 25} = \sqrt{50} \approx 7.07$，步幅较大；随着 $(x,y)$ 靠近 $(0,0)$，梯度逐渐趋近于 $\mathbf{0}$，步幅自动缩小——**梯度像一位经验丰富的向导，在陡峭处大步快走，在平坦处小心慢行**。

5. **凸函数的福利**：由于目标函数是凸函数，梯度下降法保证从任意起点收敛到唯一的全局最优解 $(0,0)$，不会陷入局部极小值。

> **一句话总结**：**梯度∇是一个向量**——它指明了函数值上升最快的方向（大小表示上升速率），负梯度方向就是下降最快的方向。梯度下降法就是**沿着负梯度向量方向、以梯度大小为步幅依据、在参数空间中不断逼近最优解的迭代过程**。等高线图直观地展示了这一几何本质：迭代路径始终沿着等高线的法线方向（即负梯度方向）前进。

## 1 梯度下降法理论、性能分析

### 1.1 梯度下降法理论及适用性分析

#### 1.1.1 梯度下降法：定义、数学证明与核心思想

梯度下降法是一种**一阶迭代优化算法**，用于寻找目标函数 $L(\theta)$ 的（局部）最小值。其核心思想是：**在函数当前点处，沿着负梯度方向前进，函数值下降最快**。

**参数更新公式**：
$$
\theta_{t+1} = \theta_t - \eta \cdot \nabla L(\theta_t)
$$

其中：
- $\theta_t$：第 $t$ 轮迭代时的参数向量
- $\eta$：**步长**，控制每次移动的距离。本文中"步长"与"学习率"为同一概念，下文将混用这两个术语。
- $\nabla L(\theta_t)$：损失函数在 $\theta_t$ 处的梯度向量

---

**为什么沿负梯度方向下降最快？——一阶泰勒展开的数学证明**

若损失函数 $L(\theta)$ 在 $\theta_t$ 处**可微**，则在 $\theta_t$ 的邻域内可展开为：
$$
L(\theta_t + \Delta \theta) = L(\theta_t) + \nabla L(\theta_t)^\top \Delta \theta + o(\|\Delta \theta\|)
$$
其中 $o(\|\Delta \theta\|)$ 是**高阶无穷小**，满足 $\lim_{\|\Delta \theta\| \to 0} \frac{o(\|\Delta \theta\|)}{\|\Delta \theta\|} = 0$。

忽略高阶无穷小，得到一阶泰勒近似：
$$
L(\theta_t + \Delta \theta) \approx L(\theta_t) + \nabla L(\theta_t)^\top \Delta \theta
$$

> ⚠️ **泰勒展开的成立条件**：上述近似**仅在 $\|\Delta \theta\|$ 充分小时成立**。

优化目标是迭代后损失降低，即满足 $L(\theta_t + \Delta \theta) < L(\theta_t)$，代入近似公式，可得核心约束条件：
$$
\nabla L(\theta_t)^\top \Delta \theta < 0
$$

注意$\nabla L(\theta_t)^\top$与$\Delta \theta$均是向量，点积$\nabla L(\theta_t)^\top\Delta \theta$是一个标量。

根据向量点积性质：
- **方向相反时（夹角 φ = 180°）**，点积取**最小值**；
- **方向成钝角时（90° < φ < 180°）**，点积也为负，但下降幅度不是最大。

因此取参数更新量 $\Delta \theta = -\eta \nabla L(\theta_t)$（$ \eta \to 0^+ $，保证$\|\Delta \theta\|$ 充分小），代入核心约束条件：
$$
\nabla L(\theta_t)^\top (-\eta \nabla L) = -\eta \|\nabla L\|^2 < 0
$$

这既严格满足 $L(\theta_{t+1}) < L(\theta_t)$ 的下降条件，采用的又是**负梯度方向是所有方向中下降幅度最大的方向**——这正是梯度下降法又称"最速下降法"的数学根源。

在实际应用中，参数更新量 $\Delta \theta = -\eta \nabla L(\theta_t)$（$\eta > 0$），$\eta$不可能无穷小，是一个**有限的、非零的步长**。**步长$\eta$ 越大，$\|\Delta \theta\|$ 就越大，高阶项 $o(\|\Delta \theta\|)$ 的误差就越显著**——负梯度方向不再是最优的迭代方向，可能产生震荡甚至发散（正如1.2.1节一维二次函数实验中 $\eta > 2.0$ 时的发散行为）。

因此，"最速下降法"是一个理论概念，被限定在 **$ \eta \to 0^+ $ 的极限**下，而非对有限步长的保证——这也正是发展出动量、Adam等变体算法的原因。

---

**"最速下降法"名称的深层澄清：瞬时最速 vs 全局最速**

理解"最速下降法"的关键，在于厘清其优化对象：

1. **优化的是"当前点"的瞬时下降率**，而非整条路径的累积下降量。
   - 方向导数 $D_{\boldsymbol u} f(\boldsymbol x_0)$ 只包含 $\boldsymbol x_0$ 这一点的梯度信息；
   - 它回答的问题是："**此时此刻**，朝哪个方向走，下降得最快？"

2. **它不对有限步长后的结果做任何保证**。
   - 负梯度方向是 $h \to 0$ 时的最优方向；
   - 对于实际采用的有限步长$\eta > 0$，函数值变化为 $\Delta f = -\eta\|\nabla f\|^2 + O(\eta^2)$；
   - $\eta$ 越大，高阶项 $O(\eta^2)$ 的影响越大，负梯度方向越可能偏离真正的"最优有限步方向"。

3. **形象类比**：
   - 梯度下降法就像站在山丘上，闭眼摸脚下的坡度（梯度），选择最陡的方向迈出一步；
   - 这一步确实是从"当前位置"出发的最陡方向，但迈出后地形已经改变，原先的方向未必是通往山脚的最优路径；
   - 每一步都是**重新基于当前点计算的"单步最优"**，而非远见全局的路线规划。

4. **非凸地形中的后果**：
   - 正是这种"近视"特性，导致梯度下降法在非凸损失面上容易陷入局部极小值或鞍点；
   - 从全局视角看，负梯度方向可能引导模型走向一个"浅坑"，而稍微偏离负梯度方向（如动量方法）反而可能越过障碍，到达更深的全局最优。

📌 **核心总结**：
 - **无穷小步长**：方向导数是极限定义，一阶泰勒近似只在 $h \to 0$ 时精确成立；
 - **最速**：在所有方向中，负梯度方向使方向导数取全局最小值 $-\|\nabla f\|$，瞬时下降速率最大；
 - **瞬时最速**：该最优性仅限于当前点的无穷小邻域，是局部瞬时最优，而非有限距离的全局路径最优。

---

**从数学原理到工程实践的桥梁：缺陷驱动的演进起点**

> 工程实践中，步长不可能无穷小，而理论上梯度下降法要求步长无穷小，这种理论与实践之间的冲突，正是后续所有优化算法改进的原动力。后续算法的演进，本质上都是在回答同一个问题：当无法满足无穷小步长条件时，如何修正梯度方向或调整步长，使更新既不过度震荡，也不在鞍点停滞？
> 
> **梯度下降法作为所有优化器的理论基座**：无论后续发展出SGD、Momentum还是Adam，它们的更新公式都可以统一写为 $\theta_{t+1} = \theta_t - \eta \cdot d_t$，其中 $d_t$ 是对原始梯度 $\nabla L$ 的各种变换——采样噪声、历史平滑、尺度归一化。每一次算法升级都围绕梯度下降法的两个核心要素展开：**梯度方向**与**步长大小**。没有梯度下降法作为参照系，就无法理解Momentum为何引入、Adam为何融合等等。
> 
> 每一次算法升级都旨在修补前代在有限步长下暴露的特定缺陷：
> - **有限步长下的震荡/停滞矛盾**（理论 $\eta \to 0$ → 工程 $\eta > 0$）→ **Momentum + AdaGrad**
> - **动量滞后超调**（Momentum → **Nesterov**）
> - **一阶动量与自适应步长的融合**（RMSprop → **Adam**）
> 
> 理解这个"瞬时最速"的数学根源，是理解后续所有优化器设计逻辑的起点：**没有算法能突破有限步长带来的误差，所有改进都是在这种限制下，对特定地形特征的补偿与折中。**

#### 1.1.2 梯度下降法的适用条件

理解GD的工作原理后，我们需要明确：梯度下降法并非万能优化器，其成功应用依赖于以下前提条件。

**核心适用条件**

| 条件 | 说明 |
|:---|:---|
| **目标函数可微** | 梯度（或次梯度）必须存在且可计算（含义详见附录B） |
| **优化问题类型** | 标准梯度下降法处理**无约束优化**；带简单约束时可用投影梯度下降法（PGD） |
| **函数值可计算** | 每次迭代能计算损失函数值，用于监控收敛 |
| **梯度可计算** | 能够通过解析求导或自动微分获得梯度向量 |

**按目标函数性质分类**

```
目标函数类型                          梯度下降法适用性与收敛保证
├── 凸函数 (Convex)                 ✅ 理论保证收敛到全局最优（含义详见附录C）
│   ├── 强凸 (Strongly Convex)          → 线性收敛速率 O(ρᵏ), 0<ρ<1
│   └── 一般凸 (General Convex)         → 次线性收敛速率 O(1/k)
│
├── 非凸函数 (Non-convex)           ⚠️ 只能保证收敛到驻点（局部极小/鞍点）
│   ├── 满足PL条件 (Polyak-Łojasiewicz) → 可证收敛到全局最优（含义详见附录E）
│   └── 一般非凸                         → 无全局最优保证，收敛点取决于初始值
│
└── 不可导/离散运算                 ❌ 标准GD完全失效
    └── 替代方案：次梯度法、坐标下降法、进化算法、强化学习
```

#### 1.1.3 深度学习是主战场

在理解梯度下降法的定义与适用条件后，一个核心问题自然浮现：**为什么深度学习——这个高度非凸、参数规模动辄百万级的问题——恰恰是梯度下降法及其变体的主战场？**

**关键点**：深度学习损失函数对参数**几乎处处可导**(只有**有限个不可导点**)，这是梯度下降法能应用于深度学习的根本前提。

展开来说，有四个层次的原因：

---

**第一层：深度学习是函数嵌套结构，仅在有限个孤立点不可导**

深度学习的本质是**多层函数的嵌套**。对于具有 $L$ 层的前馈神经网络，其输出可以递归定义为：

$$
\begin{aligned}
a_0 &= x, \\
a_l &= \sigma_l(W_l a_{l-1} + b_l), \quad l = 1, 2, \dots, L, \\
L_{\text{total}} &= \text{Loss}(y, a_L).
\end{aligned}
$$

整个系统从输入 $x$ 到损失 $L_{\text{total}}$ 构成了一条完整的计算链：

$$
x \xrightarrow{W_1,b_1} z_1 \xrightarrow{\sigma_1} a_1 \xrightarrow{W_2,b_2} z_2 \to \cdots \xrightarrow{W_L,b_L} z_L \xrightarrow{\sigma_L} a_L \xrightarrow{\text{Loss}} L_{\text{total}}
$$

这条链上的环节——**线性变换（$W_l a_{l-1} + b_l$）、激活函数（$\sigma_l$）、损失函数（Loss）**，
**某些环节存在有限个不可导点**。以 ReLU 激活函数 $\sigma(z) = \max(0, z)$ 为例：
- 作为关于输入 $z$ 的函数，ReLU 在 $z=0$ 处**不可导**（左导数0，右导数1）；


---

**第二层：为什么这些不可导点在工程上可以完全忽略？**

由于以下四个因素的共同作用，实际训练中参数**几乎永远不会恰好落在这些不可导点上**：

| 因素 | 对"恰好落在不可导点"的影响 |
|:---|:---|
| **随机初始化** | 参数从连续分布中采样，落在零测集的概率为0 |
| **Mini-batch 随机性** | 每次采样不同batch，激活值的精确位置不断变化 |
| **浮点数精度限制** | IEEE 754 浮点数无法精确表示所有实数 |
| **参数持续更新** | 训练过程中参数每步都在变化，不会固定在某一点 |

**定量直觉**：在 FP32 精度下，$[0,1]$ 区间内均匀分布的随机数恰好等于 `0.0` 的概率约为 $2.33 \times 10^{-10}$。在数十亿次的前向-反向传播中，撞上这些孤立点的次数**可以忽略不计**。

> **工程现实**：即使以极低概率撞上了不可导点，现代深度学习框架也会自动赋予一个默认的次梯度值（如 PyTorch 中 ReLU 在 0 点取 0），保证梯度信号不中断。框架甚至**不判断**是否撞到——它只是硬编码了 $z \le 0$ 取0、$z > 0$ 取1的分支逻辑。

---

**第三层：真正致命的问题不是"有限个不可导点"，而是"梯度几乎处处断裂"**

为了凸显"有限个不可导点"为什么不是问题，对比一下真正让梯度下降法失效的情况：

| 操作 | 不可导/失效情况 | 不可导点的规模 | 实际影响 |
|:---|:---|:---|
| **ReLU** $\max(0,z)$ | $z=0$ 处不可导 | **有限个孤立点**（零测集） | ✅ 完全可用 |
| **MaxPool** | 多输入平局时不可导 | **有限组合**（零测集） | ✅ 完全可用 |
| **L1 正则化** $|w|$ | $w=0$ 处不可导 | **有限个孤立点**（零测集） | ✅ 完全可用 |
| **阶跃函数** $\text{Step}(x)$ | 除0点外导数处处为0 | 导数**几乎处处为0** | ❌ 梯度信号断裂 |
| **sign(x)** | 导数几乎处处为0 | 导数**几乎处处为0** | ❌ 梯度信号断裂 |
| **argmax** | 梯度完全不存在 | 梯度**处处不存在** | ❌ 无法回传梯度 |

> ⚠️ **关键区分**：ReLU 在 0 点不可导，但这是**有限个孤立点**（零测集）；阶跃函数的导数**几乎处处为0**——前者是数学上的小瑕疵，后者是工程上的灾难。

---

**第四层：可微嵌套结构 + 线性复杂度 = 一阶梯度的双重优势**

除了"几乎处处可导"这一前提条件外，深度学习还满足梯度下降法高效运行的另外两个条件：

**可微嵌套结构保证梯度可计算**：由于整个系统是嵌套的可微函数，根据链式法则，损失对任意参数的导数几乎处处存在且可计算：

$$
\frac{\partial L_{\text{total}}}{\partial W_l} = \frac{\partial L_{\text{total}}}{\partial a_L} \cdot \frac{\partial a_L}{\partial a_{L-1}} \cdots \frac{\partial a_{l+1}}{\partial a_l} \cdot \frac{\partial a_l}{\partial W_l}
$$



**一阶梯度的线性计算复杂度**：深度学习的参数规模从百万级到千亿级不等。梯度下降法只需要**一阶梯度信息**，计算和存储成本与参数规模呈 $O(n)$ 的线性关系：

| 信息类型 | 计算复杂度 | 存储复杂度 | 能否用于大模型 |
|:---|:---|:---|
| **一阶梯度（GD）** | $O(n)$ | $O(n)$ | ✅ 可行 |
| **二阶信息（海森矩阵）** | $O(n^2)$-$O(n^3)$ | $O(n^2)$ | ❌ 不可行 |

这让梯度下降法成为大规模深度学习的**唯一实际选择**。

---

> 📌 **总结**：梯度下降法之所以能成为深度学习训练的核心算法，根本原因在于——**深度学习本质上是多层可微函数的嵌套结构，损失函数对参数只存在有限个不可导点，工程上几乎处处可导**。在这个前提之上，可微嵌套结构保证了梯度可通过链式法则高效计算，一阶梯度的线性复杂度保证了可扩展到千亿级参数。三个条件叠加，使梯度下降法在深度学习中不仅是可行的，而且是**唯一具有实际工程可行性的选择**。

> 📌 次梯度的严格数学定义及其与凸函数的关系详见附录D。

#### 1.1.5 梯度的核心地位

在深度学习中，**所有参数更新的信息来源都基于梯度**。无论采用SGD、Momentum、Adam还是任何其他优化器，它们的输入都是同一个东西——损失函数对参数的梯度。区别仅在于对梯度的加工方式不同：有的直接使用（SGD），有的对梯度做指数移动平均（Momentum），有的对梯度做自适应缩放（RMSprop），有的两者兼做（Adam）。但无论加工方式如何复杂，**梯度始终是唯一的信息源**。

这一事实意味着：深度学习的训练优化本质上是一个**以梯度为唯一输入的信息加工系统**。没有梯度，再精妙的优化算法也无从施展——反向传播无法启动，网络参数永远停留在随机初始化状态，学习根本无从发生。

反向传播的本质是链式法则，其核心工作只负责高效计算每一层权重、偏置的梯度，并不更新参数。**梯度的计算正是依赖于上述嵌套结构的逐层链式求导**：

$$
\frac{\partial L_{\text{total}}}{\partial W_l} = \frac{\partial L_{\text{total}}}{\partial a_L} \cdot \frac{\partial a_L}{\partial a_{L-1}} \cdots \frac{\partial a_{l+1}}{\partial a_l} \cdot \frac{\partial a_l}{\partial W_l}
$$

拿到梯度之后，优化器根据梯度及自身维护的动量/历史信息等状态，执行相应的参数更新策略（SGD直接使用梯度，Momentum结合历史速度，Adam结合一阶和二阶矩估计等）。

- 梯度本质是**误差信号**：从输出层向输入层回传，告诉每一层权重对最终预测错误承担多大责任，指导各层特征调整；
- 梯度接近0：参数几乎不更新，网络停止学习，对应**梯度消失**；梯度数值爆炸增大为梯度爆炸；
- ReLU、ResNet残差连接、BN、权重初始化、梯度裁剪，本质上都是在调控梯度的大小与流动，保障深度网络可训练；
- SGD/Momentum/RMSprop/AdamW全部优化器的输入都是梯度，只是对梯度做不同加工；
- 梯度同时也是对抗样本、Grad‑CAM可解释性、策略梯度等高级技术的底层基础。

### 1.2 梯度下降法的两大缺陷



#### 1.2.1 缺陷一（基础缺陷）：步长敏感——大了震荡，小了停滞

步长是决定梯度下降法成败的核心超参数。不合理的步长会直接打破泰勒近似条件，出现震荡、发散等情况。

为了从理论和实验两个层面完整揭示步长的作用机制，本节先推导一维凸二次函数上梯度下降法的误差递推闭式解，再用五组实验覆盖全部理论分类。

> **术语说明**：本节及后续内容中，"步长"与"学习率"为同一概念，均指参数更新公式中的系数 $\eta$。在强调参数"迈出多远"的几何意义时，优先使用"步长"一词。

---
 

**示例** 以标准凸二次函数作为分析对象：
$$
f(x) = 0.5x^2 - 2x
$$

其一阶导数为：
$$
f'(x) = x - 2
$$

令梯度为零，求得理论最优解：
$$
x - 2 = 0 \Rightarrow x^* = 2, \quad f(x^*) = -2
$$

梯度下降法的参数更新公式为：
$$
x_{k+1} = x_k - \eta \cdot f'(x_k)
$$

代入 $f'(x_k) = x_k - 2$：
$$
x_{k+1} = x_k - \eta(x_k - 2) = (1 - \eta)x_k + 2\eta
$$

为了分析迭代点 $x_k$ 与最优解 $x^*$ 之间的距离如何变化，定义误差项 $e_k = x_k - x^*$。将 $x_k = e_k + x^*$ 代入上式：

$$
\begin{aligned}
e_{k+1} + x^* &= (1 - \eta)(e_k + x^*) + 2\eta \\
e_{k+1} &= (1 - \eta)e_k + (1 - \eta)x^* + 2\eta - x^* \\
         &= (1 - \eta)e_k + \eta(2 - x^*)
\end{aligned}
$$

由于 $x^* = 2$，常数项 $\eta(2 - x^*) = 0$ 消去，得到简洁的误差递推式：
$$
e_{k+1} = (1 - \eta) \cdot e_k
$$

从 $k=0$ 开始累乘展开，得到闭式解：
$$
e_k = (1 - \eta)^k \cdot e_0
$$

该公式将步长$\eta$ 与收敛行为之间的关联从经验观察升级为精确的数学定理——收敛性完全由误差递推因子 $|1 - \eta|$ 决定：

| 步长范围 | 误差递推因子 | 误差变化 | 迭代行为 |
|:---|:---|:---|:---|
| $0 < \eta < 1$ | $\|1 - \eta\| < 1$ | 指数衰减 | **单调收敛** |
| $\eta = 1$ | $\|1 - \eta\| = 0$ | 一步归零 | **一步收敛** |
| $1 < \eta < 2$ | $\|1 - \eta\| < 1$ | 正负交替衰减 | **震荡收敛** |
| $\eta = 2$ | $\|1 - \eta\| = 1$ | 绝对值恒定 | **等幅震荡，永不收敛** |
| $\eta > 2$ | $\|1 - \eta\| > 1$ | 指数增长 | **发散** |

$\eta = 2$ 是收敛与发散之间精确的临界分界点。

注意：现实中的深度学习损失函数——非凸、高维、不可解析——我们无法像上述推导那样精确计算收敛临界点，只能通过步长调度、自适应优化器（Adam/RMSprop）、梯度裁剪等手段来规避超调/发散风险，在可控范围内尽可能加快收敛。


---

**实验验证**

为完整验证上述五种理论分类，本节选取五个代表性步长，覆盖表格中的全部行为模式：

| 实验编号 | 步长$\eta$ | 对应理论分类 | 预期迭代行为 |
|:---|:---|:---|:---|
| 实验1 | $\eta = 0.6$ | $0 < \eta < 1$ | 单调收敛 |
| 实验2 | $\eta = 1.0$ | $\eta = 1$ | 一步收敛 |
| 实验3 | $\eta = 1.5$ | $1 < \eta < 2$ | 震荡收敛 |
| 实验4 | $\eta = 2.0$ | $\eta = 2$ | 等幅震荡，永不收敛 |
| 实验5 | $\eta = 2.4$ | $\eta > 2$ | 发散 |

所有实验共用目标函数 $f(x)=0.5x^2-2x$ 和初值 $x_0=-4$。

**实验1：单调收敛（η = 0.6）**

当步长$0 < \eta < 1$ 时，误差递推因子 $|1 - \eta| < 1$，误差以指数方式衰减。迭代点从 $x_0 = -4$ 出发，沿单一方向稳步靠近最优解 $x^* = 2$，坐标符号不翻转，全程**没有震荡**。

从误差递推公式 $e_k = (1 - \eta)^k e_0$ 可知，当 $\eta = 0.6$ 时，$e_k = (0.4)^k e_0$，每步误差缩小为前一步的 40%，呈现稳定、平滑的收敛轨迹。

> **典型应用场景**：当目标函数地形较为平坦、不需要快速收敛时，较小的步长可确保稳定的迭代过程。

In [ ]:
# ===== 导入必要的库 =====
import numpy as np                      # 数值计算库
import plotly.graph_objects as go      # 交互式可视化库

# ========= 定义目标函数及其导数 =========
def f(x):
    """
    目标函数：f(x) = 0.5 * x^2 - 2x，凸二次函数。
    这是一个开口向上的抛物线，极值点在导数等于0的地方。
    """
    return 0.5 * x**2 - 2 * x   # 返回函数值：0.5x² - 2x

def df(x):
    """
    目标函数的一阶导数：f'(x) = x - 2。
    梯度（导数）指向函数值增长最快的方向，梯度下降法就是沿负梯度方向寻找极值。
    """
    return x - 2   # 返回导数值：x - 2

# ========= 理论最优解 =========
x_star = 2.0                              # 令 f'(x)=x-2=0 解得 x=2，这是理论最优点
y_star = f(x_star)                       # 计算理论极小值：f(2) = 0.5*(4) - 2*2 = 2 - 4 = -2
print(f"【理论最优解】x* = {x_star:.2f}, f(x*) = {y_star:.2f}\n")  
# 打印理论最优解，保留两位小数

# ========= 辅助函数：判断收敛类型 =========
def get_convergence_type(eta, xs):
    """
    根据学习率 eta 自动判断收敛类型（基于理论分类）。
    参数：
        eta: 学习率
        xs: 迭代序列
    返回：
        (类型标签, 描述字符串)
    """
    # 根据理论分类表确定收敛类型
    if eta == 0.6:
        return "单调收敛", "0 < η < 1 单调收敛"
    elif eta == 1.0:
        return "一步收敛", "η = 1 一步收敛"
    elif eta == 1.5:
        return "振荡收敛", "1 < η < 2 振荡收敛"
    elif eta == 2.0:
        return "等幅振荡", "η = 2 等幅振荡，永不收敛"
    elif eta == 2.4:
        return "发散", "η > 2 发散"
    else:
        # 通用判断逻辑（备用）
        if eta < 1.0:
            return "单调收敛", f"0<η<1 单调收敛"
        elif eta == 1.0:
            return "一步收敛", "η=1 一步收敛"
        elif 1.0 < eta < 2.0:
            return "振荡收敛", "1<η<2 振荡收敛"
        elif eta == 2.0:
            return "等幅振荡", "η=2 等幅振荡，永不收敛"
        else:
            return "发散", "η>2 发散"

# ========= 通用函数：运行梯度下降并绘图 =========
def run_gd(x0, eta, max_iter=30):
    """
    运行梯度下降法并打印迭代过程。
    参数：
        x0: 初始点
        eta: 学习率（步长）
        max_iter: 最大迭代次数，默认30次
    返回：
        xs: 所有迭代点的x坐标构成的NumPy数组
        ys: 所有迭代点的函数值f(x)构成的NumPy数组
        eta: 学习率
        converged: 是否收敛到最优解附近
        convergence_type: 收敛类型标签
        convergence_desc: 收敛类型描述
    """
    xs = [x0]                      # 初始化列表，记录所有迭代点的x坐标，加入初始点
    ys = [f(x0)]                   # 初始化列表，记录所有迭代点的函数值，加入初始点的函数值
    
    x_current = x0                 # 设置当前点为初始点
    
    print(f"\n========== η={eta:.1f} ==========")  # 打印当前学习率的分隔标题
    
    converged = False              # 初始化收敛标志
    
    for k in range(max_iter):      # 开始迭代循环，k从0到max_iter-1
        g = df(x_current)          # 计算当前位置的导数（梯度）
        
        x_next = x_current - eta * g  
        # 梯度下降更新公式：x_new = x_old - 学习率 * 梯度
        
        xs.append(x_next)          # 将新位置x加入坐标列表
        ys.append(f(x_next))       # 将新位置对应的函数值加入值列表
        
        # 打印迭代步骤的详细计算过程
        print(f"x_{k}={x_current:.1f} ⇒ x_{k+1}=x_{k}‑η·y' = {x_current:.1f}‑{eta:.1f}·({g:.1f})={x_next:.1f};")
        
        if abs(g) < 0.001:         # 判断是否满足提前终止条件：梯度的绝对值小于0.001
            converged = True       # 标记为已收敛
            break                  # 如果满足（接近极小值），跳出循环
        
        x_current = x_next         # 更新当前点
    
    # 获取收敛类型
    conv_type, conv_desc = get_convergence_type(eta, np.array(xs))
    
    return np.array(xs), np.array(ys), eta, converged, conv_type, conv_desc

# 创建一个字典，用于将阿拉伯数字转换为带下标的Unicode字符（如 "x0" 转为 "x₀"）
digit_sub = {"0":"₀","1":"₁","2":"₂","3":"₃","4":"₄",
             "5":"₅","6":"₆","7":"₇","8":"₈","9":"₉"}

# ========= 绘图函数1：迭代轨迹图 =========
def plot_trajectory(xs, ys, eta, convergence_type, convergence_desc, converged, experiment_num):
    """
    绘制单变量函数的梯度下降轨迹图。
    参数：
        xs: 迭代过程中的x坐标数组
        ys: 迭代过程中的函数值数组
        eta: 学习率（用于在标题中标识）
        convergence_type: 收敛类型标签
        convergence_desc: 收敛类型描述
        converged: 是否收敛
        experiment_num: 实验编号
    """
    # ====== 自动生成标题 ======
    # 构建收敛状态描述
    status = "✓ 收敛" if converged else "… 未收敛"
    
    # 构建标题：包含实验编号、eta值、收敛类型和描述
    title_text = f"实验{experiment_num}: η={eta:.1f} | {convergence_desc}"
    if converged:
        title_text += f" | {status}"
    
    # ====== 计算绘图的范围 ======
    x_min, x_max = np.min(xs), np.max(xs)  
    # 获取迭代路径中x坐标的最小值和最大值
    
    x_pad = 0.15 * abs(x_max - x_min) if abs(x_max - x_min) > 0.1 else 2.0  
    # 计算x轴两侧的边距
    
    x_plot_low = x_min - x_pad      # 设置x轴绘图下限
    x_plot_high = x_max + x_pad     # 设置x轴绘图上限
    
    x_curve = np.linspace(x_plot_low, x_plot_high, 2000)  
    # 在绘图范围内生成2000个均匀分布的点
    
    y_curve = f(x_curve)            # 计算这些点对应的函数值

    # ====== 创建Plotly图形 ======
    fig = go.Figure()               # 初始化一个空的Plotly图形对象
    
    # === 目标函数曲线：黑色 ===
    fig.add_trace(go.Scatter(
        x=x_curve, 
        y=y_curve, 
        mode="lines",
        line=dict(color="black", width=2), 
        name="y=0.5x²−2x"
    ))
    
    # === 迭代路径连线：红色（仅第一条线显示图例，避免重复） ===
    for i in range(len(xs)-1):      # 循环遍历所有相邻的迭代点对
        show_legend = (i == 0)      # 只在第一条线段上显示图例
        fig.add_trace(go.Scatter(
            x=[xs[i], xs[i+1]], 
            y=[ys[i], ys[i+1]],
            mode="lines", 
            line=dict(color="red", width=1.2),
            name="迭代路径" if show_legend else None,
            legendgroup="迭代路径" if show_legend else None,
            showlegend=show_legend
        ))
    
    # === 迭代点marker：红色（单独一个trace，用于图例中显示"迭代点"） ===
    fig.add_trace(go.Scatter(
        x=xs, 
        y=ys, 
        mode="markers",
        marker=dict(color="red", size=4),
        name="迭代点",
        legendgroup="迭代点",
        showlegend=True
    ))

    # ====== 为迭代点添加文本标注 ======
    annot_list = []                 # 初始化一个空列表
    
    for idx, (xi, yi) in enumerate(zip(xs, ys)):  
        y_shift = 8 if idx % 2 == 0 else -10  
        # 交错向上/向下移动文本，避免标注重叠
        
        idx_str = str(idx)          # 将索引转换为字符串
        sub_text = "".join([digit_sub[c] for c in idx_str])  
        # 将索引字符转换为带下标的Unicode字符
        
        # 标注文本：红色，保持整体一致性
        annot_list.append(dict(
            x=xi, 
            y=yi, 
            text=f"x{sub_text}", 
            showarrow=False,
            xanchor="left", 
            yshift=y_shift, 
            font=dict(size=8, color="red")
        ))
    
    # ====== 更新图表布局 ======
    fig.update_layout(
        annotations=annot_list,     # 将标注列表添加到图形中
        title=dict(text=title_text, font=dict(size=14)),  # 设置标题和字体大小
        xaxis_title="x",            # x轴标题
        yaxis_title="y",            # y轴标题
        template="plotly_white",    # 使用干净的白色背景模板
        width=1100, height=600,     # 设置图表的宽高
        hovermode="x unified",      # 鼠标悬停时，统一显示同一x坐标下的所有信息
        showlegend=True,            # 显示图例
        legend=dict(
            orientation="h",        # 图例水平排列
            yanchor="bottom",       # 图例底部对齐
            y=1.02,                 # 图例放在图表上方
            xanchor="center",       # 图例水平居中
            x=0.5,
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="lightgray",
            borderwidth=1
        )
    )
    fig.show()                      # 显示交互式图形

# ========= 绘图函数2：收敛曲线图（无图例、无标注） =========
def plot_convergence(xs, ys, eta, convergence_type, convergence_desc, converged, experiment_num):
    """
    绘制梯度下降的收敛曲线图（函数值随迭代次数的变化）。
    参数：
        xs: 迭代过程中的x坐标数组
        ys: 迭代过程中的函数值数组
        eta: 学习率（用于在标题中标识）
        convergence_type: 收敛类型标签
        convergence_desc: 收敛类型描述
        converged: 是否收敛
        experiment_num: 实验编号
    """
    # ====== 自动生成标题 ======
    # 构建收敛状态描述
    status = "✓ 收敛" if converged else "… 未收敛"
    
    # 构建标题
    title_text = f"实验{experiment_num}: η={eta:.1f} | {convergence_desc}"
    if converged:
        title_text += f" | {status}"
    
    # ====== 创建Plotly图形 ======
    fig = go.Figure()
    
    iterations = np.arange(len(ys))  # 迭代次数：0, 1, 2, ...
    
    # === 收敛曲线：函数值随迭代次数的变化 ===
    fig.add_trace(go.Scatter(
        x=iterations, 
        y=ys, 
        mode="lines+markers",
        line=dict(color="blue", width=2.5),
        marker=dict(color="blue", size=6, symbol="circle"),
        name="收敛曲线",
        showlegend=False           # 不显示图例
    ))
    
    # === 设置坐标轴 ===
    fig.update_xaxes(
        title_text="迭代次数 k",
        tickmode="linear",
        dtick=1,                   # 每1个迭代步长显示刻度
        gridcolor="lightgray",
        gridwidth=0.5
    )
    
    fig.update_yaxes(
        title_text="f(x)",
        gridcolor="lightgray",
        gridwidth=0.5
    )
    
    # ====== 更新图表布局（不显示图例） ======
    fig.update_layout(
        title=dict(
            text=title_text, 
            font=dict(size=14)
        ),
        template="plotly_white",
        width=1000, 
        height=600,
        hovermode="x unified",
        showlegend=False,          # 全局关闭图例
        margin=dict(l=60, r=60, t=80, b=60)  # 调整边距
    )
    fig.show()                      # 显示交互式图形


# ========= 主程序：按截图依次运行5个实验 =========

# 定义实验配置列表：[(实验编号, 初始点, 学习率, 最大迭代次数)]
experiments = [
    (1, -4.0, 0.6, 30),   # 实验1：η = 0.6，单调收敛
    (2, -4.0, 1.0, 30),   # 实验2：η = 1.0，一步收敛
    (3, -4.0, 1.5, 30),   # 实验3：η = 1.5，振荡收敛
    (4, -4.0, 2.0, 30),   # 实验4：η = 2.0，等幅振荡，永不收敛
    (5, -4.0, 2.4, 30),   # 实验5：η = 2.4，发散
]

# 循环运行所有实验
for exp_num, x0, eta, max_iter in experiments:
    print(f"\n{'='*50}")
    print(f"开始 实验{exp_num}: η = {eta}")
    print(f"{'='*50}")
    
    # 运行梯度下降
    xs, ys, eta, converged, conv_type, conv_desc = run_gd(x0, eta, max_iter)
    
    # 绘制迭代轨迹图
    plot_trajectory(xs, ys, eta, conv_type, conv_desc, converged, exp_num)
    
    # 绘制收敛曲线图
    plot_convergence(xs, ys, eta, conv_type, conv_desc, converged, exp_num)
    
    print(f"\n实验{exp_num} 完成！")

print("\n" + "="*50)
print("所有5个实验已完成！")
print("="*50)

**实验结果解读**

五组实验的结果与误差递推公式 $e_k = (1-\eta)^k e_0$ 的预测完全一致：

1. **η=0.6（$0<\eta<1$）**：$|1-\eta|=0.4<1$，误差指数衰减。迭代序列从 $x_0=-4$ 单向稳步靠近 $x^*=2$，坐标符号不翻转，全程**没有震荡**；

2. **η=1.0（$\eta=1$）**：$|1-\eta|=0$，误差一步归零。$x_1 = (1-1)\cdot(-4)+2\cdot1 = 2$，**一步直达最优解**；

3. **η=1.5（$1<\eta<2$）**：$|1-\eta|=0.5<1$，误差正负交替衰减。迭代点在极小点 $x^*=2$ 两侧来回跨越，振幅逐步衰减，最终收敛——**震荡收敛**；

4. **η=2.0（$\eta=2$）**：$|1-\eta|=1$，误差绝对值恒定。迭代点严格在 $-4 \leftrightarrow 8$ 两点来回跳，**等幅震荡，永远到不了极小点**；

5. **η=2.4（$\eta>2$）**：$|1-\eta|=1.4>1$，误差绝对值指数增长。每一步偏移幅度越来越大，离最优解越来越远，损失持续上升，**完全发散**。

这个实验清晰证明：哪怕是完美凸损失，步长设置不当也会震荡甚至发散。GD的迭代对步长敏感是普遍存在的理解这一点，就理解了为什么后续所有优化算法都在步长调控上投入了巨大的设计精力。

---

**步长敏感的根源**：梯度下降法的更新公式 $\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t)$ 中，步长 $\eta$ 是一个**全局统一的标量**。这意味着：

- 在所有迭代步数上，$\eta$ 保持不变（无调度的情况下）
- 在所有参数维度上，$\eta$ 完全一致

**这带来了两个层面的问题**：

| 层面 | 问题 | 后果 |
|:---|:---|:---|
| **时序层面** | 训练初期需要大步长快速下降，训练后期需要小步长精细收敛 | 固定步长无法同时满足——初期太慢或后期震荡 |
| **空间层面** | 不同参数维度的曲率差异巨大（陡峭方向 vs 平缓方向） | 统一步长导致陡峭方向震荡、平缓方向停滞 |

⚠️ **步长敏感是GD的基础缺陷，这个问题是广泛存在的数学论证，详见附录。

**这正是为什么后续的AdaGrad、RMSprop、Adam等算法要在步长调控上投入大量设计精力——它们试图解决的就是"有限步长下如何既快又稳"这个根本矛盾。**

#### 1.2.2 缺陷二（根本缺陷）：方向短视——局部梯度决策，病态曲率下震荡低效

如果说步长敏感是GD的"基础缺陷"，那么方向短视则是GD更根本的缺陷——它源于GD的更新方向**完全由当前点的局部梯度决定**，缺乏对地形全局结构的感知。


In [ ]:
# 病态二次函数：GD法在峡谷中的锯齿震荡演示
# ==================================================
# 目标函数：f(x, y) = 0.5 * (x^2 + 100 * y^2)
# 条件数 κ = λmax/λmin = 100/1 = 100
#   · x方向的二阶导数为1（曲率小，地形平缓，等高线稀疏）
#   · y方向的二阶导数为100（曲率大，地形陡峭，等高线密集）
# 全局最优：(0, 0)，f(0,0) = 0
#
# 实验目的：
# 展示GD法在病态条件下的锯齿震荡困境，引出动量法的必要性。

import numpy as np
import plotly.graph_objects as go


# ============================================================
# 运行参数（统一在此设置）
# ============================================================

# 起点坐标：x远离最优（-9），y接近最优（0.1）
# 这样设置是为了突出x方向（平缓方向）的收敛瓶颈
START_POINT = (-9.0, 0.1)

# 学习率：略小于y方向稳定上限 2/λmax = 2/100 = 0.02
# 取0.02刚好在稳定边界上，保证GD法在y方向不发散
LEARNING_RATE = 0.0198

# 迭代步数：100步，足够展示锯齿震荡现象
NUM_STEPS = 100

# 可视化参数
X_RANGE = [-10, 10]        # x轴显示范围
Y_RANGE = [-1.5, 1.5]      # y轴显示范围
CONTOUR_START = 0.1        # 等高线起始值
CONTOUR_END = 60           # 等高线终止值
CONTOUR_SIZE = 3           # 等高线间隔
FIGURE_HEIGHT = 800        # 图表高度
COLORMAP = "Viridis"       # 颜色方案


# ============================================================
# 目标函数与梯度定义
# ============================================================

def f_ill(x, y):
    """
    病态二次函数 f(x, y) = 0.5 * (x² + 100·y²)
    
    Hessian矩阵：H = [[1, 0], [0, 100]]
    特征值：λmin = 1（x方向），λmax = 100（y方向）
    条件数：κ = 100（高度病态）
    
    参数：
        x, y: 二维坐标点
    返回：
        函数值
    """
    return 0.5 * (x**2 + 100 * y**2)


def grad_f_ill(x, y):
    """
    梯度：∂f/∂x = x, ∂f/∂y = 100·y
    
    参数：
        x, y: 二维坐标点
    返回：
        (∂f/∂x, ∂f/∂y)
    """
    return x, 100 * y


# ============================================================
# GD法实现
# ============================================================

def vanilla_gd_ill(x0, y0, lr, steps):
    """
    GD法（标准梯度下降法）
    
    更新公式：x_{t+1} = x_t - lr * gradient
    
    参数：
        x0, y0: 起始坐标
        lr: 学习率
        steps: 迭代步数
    返回：
        numpy数组，包含每步的坐标路径
    """
    x, y = x0, y0
    path = [(x, y)]
    
    for _ in range(steps):
        gx, gy = grad_f_ill(x, y)
        x = x - lr * gx
        y = y - lr * gy
        path.append((x, y))
    
    return np.array(path)


# ============================================================
# 运行实验
# ============================================================

path_gd_ill = vanilla_gd_ill(
    START_POINT[0], 
    START_POINT[1], 
    lr=LEARNING_RATE, 
    steps=NUM_STEPS
)


# ============================================================
# 生成等高线地形数据
# ============================================================

xs_ill = np.linspace(X_RANGE[0], X_RANGE[1], 600)
ys_ill = np.linspace(Y_RANGE[0], Y_RANGE[1], 600)
X_ill, Y_ill = np.meshgrid(xs_ill, ys_ill)
Z_ill = f_ill(X_ill, Y_ill)


# ============================================================
# 绘制等高线填充图 + GD优化路径
# ============================================================

fig_ill = go.Figure()

# 等高线填充图
fig_ill.add_trace(go.Contour(
    x=xs_ill, 
    y=ys_ill, 
    z=Z_ill,
    colorscale=COLORMAP,
    contours=dict(
        coloring="fill",
        showlabels=True,
        labelfont=dict(size=10, color="white"),
        start=CONTOUR_START, 
        end=CONTOUR_END, 
        size=CONTOUR_SIZE
    ),
    line=dict(width=0.8, color="white"),
    showscale=True,
    colorbar=dict(title="f(x,y)", thickness=20),
    opacity=0.85,
    name="损失地形"
))

# GD法迭代路径
fig_ill.add_trace(go.Scatter(
    x=path_gd_ill[:, 0], 
    y=path_gd_ill[:, 1],
    mode="lines+markers",
    line=dict(color="red", width=1.5),
    marker=dict(size=3, color="red"),
    name=f"GD法 (lr={LEARNING_RATE}, {NUM_STEPS}步)"
))

# 起点标记
fig_ill.add_trace(go.Scatter(
    x=[START_POINT[0]], 
    y=[START_POINT[1]],
    mode="markers",
    marker=dict(
        size=8, 
        color="red"
    ),
    name="起点"
))

# 全局最优标记
fig_ill.add_trace(go.Scatter(
    x=[0], 
    y=[0],
    mode="markers",
    marker=dict(
        size=10, 
        color="red", 
        symbol="star"
    ),
    name="全局最优"
))

# 图表布局
fig_ill.update_layout(
    height=FIGURE_HEIGHT,
    template="plotly_white",
    title=dict(
        text="GD：锯齿震荡迭代",
        x=0.12,                   # 标题居中
        xanchor="center",
        font=dict(size=20)
    ),
    xaxis_title="x（低曲率/平缓方向）",
    yaxis_title="y（高曲率/陡峭方向）",
    xaxis=dict(range=X_RANGE),
    yaxis=dict(range=Y_RANGE),
    legend=dict(
        orientation="h", 
        yanchor="bottom", 
        y=1.02,
        xanchor="center",         # 图例居中
        x=0.5,
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="lightgray", 
        borderwidth=1,
        font=dict(size=13)
    ),
    margin=dict(t=100, b=80, l=80, r=80)
)

fig_ill.show()


# ============================================================
# 打印收敛结果
# ============================================================

print("=" * 70)
print("【病态二次函数 κ=100：GD法的困境】")
print(f"学习率 α={LEARNING_RATE}，迭代步数 {NUM_STEPS}")
print("=" * 70)
print(f"起点: ({START_POINT[0]}, {START_POINT[1]}), f={f_ill(*START_POINT):.4f}")
print("-" * 70)

print(f"GD法  终点: x={path_gd_ill[-1,0]:.8f}, y={path_gd_ill[-1,1]:.8f}")
print(f"         f = {f_ill(*path_gd_ill[-1]):.10f}")
print("-" * 70)

print("【震荡行为分析】")
print(f"  · x方向（平缓）：梯度小，每步仅前进约 {LEARNING_RATE:.3f} * x")
print(f"  · y方向（陡峭）：梯度大，每步更新约 {LEARNING_RATE:.3f} * 100 * y")
print(f"  · 学习率 {LEARNING_RATE} 受限于 y 方向稳定上限 (2/100 = 0.02)")
print("  · 导致在 x 方向收敛缓慢，路径呈'之'字形震荡")
print("=" * 70)

造成低效震荡的原因是"方向短视"，其根源可以归结为：**负梯度方向的最优性，仅在步长 $\eta \to 0$ 的极限下才严格成立。一旦步长不为无穷小（工程中的必然选择），这个"最优性"就失效了。**

为了把这个逻辑链条说透，我们可以拆解为三层递进的论证：

---

### 第一层：数学根源（泰勒展开的"有效期"）

在正文的 1.1.1 节中，你已经看到了最速下降法的数学证明。我们把这个证明再往前推一步，看看步长 $\eta$ 在其中扮演的角色。

对于损失函数 $L(\theta)$，在 $\theta_k$ 处做一阶泰勒展开：

$$L(\theta_k - \eta \nabla L) = L(\theta_k) - \eta \|\nabla L\|^2 + O(\eta^2)$$

- **当 $\eta \to 0$ 时**：$O(\eta^2)$ 是比 $\eta$ 高阶的无穷小，可以忽略不计。此时，负梯度方向确实是**唯一**能让函数值下降最多的方向——**理论上的"最速"**。
- **当 $\eta > 0$ 时**：$O(\eta^2)$ 项**不可忽略**。这个高阶项里包含了海森矩阵（Hessian）的信息（即曲率）。

**结论**：只要 $\eta$ 是有限的（工程中必须如此），负梯度方向就不再是数学上精确的"最速下降方向"。这就是"方向短视"的数学源头——**梯度只提供了当前点的局部切线信息，而有限步长迫使算法必须考虑邻域内的曲率，但梯度本身对此一无所知。**

---

### 第二层：地形放大（病态曲率下的"锯齿"灾难）

既然有限步长下方向不再最优，那么当损失面地形糟糕时，这个误差会被急剧放大。

在病态曲率（长窄峡谷）地形中，海森矩阵的条件数 $\kappa = \frac{\lambda_{\max}}{\lambda_{\min}} \gg 1$。结合第一层的泰勒展开，我们可以推导出：**为了让算法在陡峭方向（$\lambda_{\max}$）不震荡，步长被迫设得很小（$\eta < 2/\lambda_{\max}$），但这导致平缓方向（$\lambda_{\min}$）几乎不动。**

更致命的是方向问题：

- 在峡谷中，当前点的**精确负梯度**指向峡谷对岸（横向）。
- 因为步长有限，这一步**一定会跨过峡谷底部，撞到对岸**。
- 到了对岸，新的负梯度又指向**原方向的对岸**。

这就形成了你看到的**锯齿震荡**。震荡的本质就是：**有限的步长 + 仅依赖当前点的梯度决策 = 反复"矫枉过正"**。

> **例证**：附录 G-4 的病态曲率演示清晰地展示了这一点——GD 的路径呈"之"字形，有效前进距离（沿谷底方向）远远小于实际移动距离（横向震荡）。

---

### 第三层：缺陷的"根本性"在于信息缺失

之所以称其为**根本缺陷**，而不是像"步长敏感"那样的基础缺陷，是因为它触及了**信息源**的问题：

| 缺陷类型 | 问题根源 | 修补方式 |
| :--- | :--- | :--- |
| **步长敏感** | $\eta$ 是全局标量，无法适配不同方向的曲率。 | 引入**二阶信息**（如 AdaGrad/RMSprop 用梯度平方缩放 $\eta$）。 |
| **方向短视** | 更新方向仅依赖**当前点**的梯度，不包含历史轨迹或前方地形的预判。 | 引入**历史信息**（如 Momentum 累积之前的梯度方向来平滑震荡）。 |

**换句话说**：

> 梯度下降法就像一个**蒙着眼、仅凭脚下瞬时坡度**走路的人。只要他迈出的步子是有限的（不可能无限小），他就不可能只靠"脚下的坡度"完美地走下山谷——他必然会因为无法预判"下一步的地形"（曲率）而走错方向（震荡）。这正是"最速下降法"这个名称在工程中失效的根本原因，也是动量（Momentum）和自适应学习率（Adam）等算法要引入额外"记忆"或"感知"机制的根本动机。

---

### 总结


**"梯度指明了无穷小步长下的最优方向，但工程中使用的有限步长，使得这个方向注定是'短视'的；当曲率病态时，这种短视就暴露为致命的震荡。"**



**这正是Momentum等算法要解决的核心问题**：通过累积历史梯度来平滑方向、压制震荡，让优化器在沟壑地形中能够更稳定地沿谷底前进。

> ⚠️ **方向短视是GD的根本缺陷**。它不依赖于步长选择——即使步长完美，GD在病态曲率地形中仍然会走"之"字形路径。步长敏感可以通过调参缓解，方向短视则需要引入新机制（动量、二阶信息等）才能改善。

#### 1.2.3 从GD缺陷到优化器演进的逻辑链条

GD的两大根本缺陷，精准对应了后续优化器的改进方向：

| GD的根本缺陷 | 对应的改进算法 | 改进机制 |
|:---|:---|:---|
| **步长敏感**（统一η无法适配不同曲率） | AdaGrad / RMSprop / Adam | 每个参数独立的自适应步长 |
| **方向短视**（病态曲率下锯齿震荡，路径低效） | Momentum / Nesterov / Adam | 累积历史梯度，平滑方向，压制震荡 |
| **两者叠加** | **Adam** | 动量（方向修正）+ 自适应步长（步长修正） |

优化器的演进史，本质上是一部**对GD这两大缺陷的修补史**。

## 2 梯度下降法改进变体：从GD到Adam的演化

> **⚠️ 核心铺垫：为什么梯度下降法需要改进？**
>
> 在深入具体算法之前，我们必须先理解一个根本矛盾：
>
> **理论要求**：梯度下降法要保证"最速下降"和稳定性，需要 $\eta \to 0$（无穷小步长），因为一阶泰勒展开只在 $\|\Delta \theta\| \to 0$ 时精确成立。
>
> **工程现实**：我们不可能用无穷小步长——那意味着永远收敛不到最优解。必须使用有限的 $\eta > 0$。
>
> **这就是所有优化器变体诞生的根源**：当 $\eta$ 必须有限时，标准GD在以下两个维度上暴露缺陷——
>
> | 问题维度 | 具体表现 | 后果 |
> |:---|:---|:---|
> | **方向问题** | 有限步长下，负梯度方向不再是真正的最速方向；在沟壑地形中，梯度方向剧烈变化导致锯齿震荡 | 收敛路径震荡，效率低下 |
> | **步长问题** | 不同维度曲率差异大，统一 $\eta$ 无法同时满足：陡峭方向需要小步长（否则震荡），平缓方向需要大步长（否则停滞） | 陡峭方向震荡，平缓方向停滞 |
>
> **后续所有优化算法，本质上都是对这两个问题的修补。** 每一次算法升级，都在回答同一个问题："当 $\eta$ 不能无穷小时，我们怎么调整方向和步长，才能既快又稳地到达最优解——从单步最优到全局较优？"
>
> | 缺陷 | 修补算法 |
> |:---|:---|
> | 方向震荡 → **Momentum** | 累积历史梯度，平滑方向，压制震荡 |
> | 统一步长 → **AdaGrad/RMSprop** | 每个维度独立步长，陡峭用小步长，平缓用大步长 |
> | 动量超调 → **Nesterov** | 先"探头"看前方，再修正方向 |
> | 方向+步长 → **Adam** | 动量（方向修正）+ 自适应步长（步长修正） |
>
> 掌握了这个"缺陷驱动"的逻辑链条，就能理解所有优化器的设计动机和适用场景。

### 2.1 工程基石：Mini-batch SGD（随机梯度下降法）

在理解算法改进之前，必须先区分三个层次的概念：

```
层次一：理论原点——全量梯度下降法（Full-Batch GD）
    ├── 用全部样本计算精确梯度
    ├── "最速下降法"这个名称属于它（无穷小步长极限下）
    └── 每步迭代计算量大，无法扩展到大数据集

        ↓ 工程改进（采样策略变化，更新公式未变）

层次二：工程实现——Mini-batch SGD
    ├── 用随机采样的一个batch估计梯度（梯度的无偏估计）
    ├── 更新公式仍是 θ = θ - η·g，与GD完全一致
    ├── 计算成本大幅降低，GPU并行加速（一次矩阵运算处理整个batch）
    ├── 随机噪声有助于逃离局部极小（详见"常见误区"章节）
    └── "最速下降法"这个名称不适用于SGD

        ↓ 真正的算法改进（修改更新公式本身）

层次三：算法改进——Momentum / AdaGrad / RMSprop / Adam
    ├── 引入新机制（动量累积、自适应步长等）
    └── 更新公式不再是 θ = θ - η·g
```

> **关键区分**：从全量GD到Mini-batch SGD是**工程实现层面的改进**——改变了梯度的计算方式（用子集估计代替全量计算），但更新公式 $\theta_{t+1} = \theta_t - \eta \cdot g$ 完全不变。真正对GD算法本身进行改进的是Momentum、AdaGrad、Adam等——它们修改了更新公式，引入了新机制（历史动量、自适应步长等）。

---

#### Mini-batch SGD 的核心特征

- **原理**：`θ = θ - lr * g`，其中 `g` 是当前 mini-batch 的梯度估计。

- **优点**：
  - 计算成本低：每步只用少量样本
  - **GPU并行加速**：batch内样本的梯度计算相互独立，可通过一次矩阵运算 `X @ W` 并行完成整个batch的前向和反向传播。用接近单样本2倍的时间获得256个样本的梯度信息，这是SGD取代全量GD的核心工程动因。
  - **随机梯度噪声有助于逃离局部极小**：SGD的梯度估计噪声提供了随机扰动，使得参数有机会从局部极小区域"晃出来"。但需注意，这只是统计层面的经验现象，并无理论保证（详见"常见误区"章节）。

- **缺点**：
  - 收敛路径可能震荡，对步长选择敏感
  - 在梯度方向变化剧烈的"沟壑"地形中前进缓慢
  - 所有参数共享同一个全局步长（详见2.1.1节）

> **名称澄清**："最速下降法"这个称呼仅属于全量GD（在无穷小步长极限下），Mini-batch SGD不使用这个名称。

### 2.1.1 从"一刀切"到"千人千面"：个性化步长的必要性

Mini-batch SGD（以及基础GD）的一个深层缺陷是：**所有参数共享一个全局步长**。在复杂的高维优化问题中，这种"一刀切"的策略严重限制了收敛效率。

#### 直观理解：狭长山谷的困境

想象一个狭长的山谷，x轴方向极其陡峭，y轴方向非常平缓。

- 在**陡峭方向（x轴）**：梯度很大。如果步长太大，参数会像喝醉了一样在山谷两侧来回剧烈震荡，永远无法稳定下来。
- 在**平缓方向（y轴）**：梯度很小。如果步长太小，参数在平缓方向上挪动极其缓慢，需要极多的迭代才能到达谷底。

使用全局步长时，我们被迫做出痛苦的妥协：
- 为了不在陡峭方向震荡，必须把步长设得很小。
- 结果在平缓方向，这个很小的步长导致前进极其缓慢——这正是经典SGD在"沟壑"地形中效率低下的根本原因。

**解决方案**：给每个参数分配它自己的步长。
- 对于**梯度大（陡峭）** 的参数，给它一个**小步长**，防止震荡。
- 对于**梯度小（平缓）** 的参数，给它一个**大步长**，加速前进。

这就是AdaGrad和RMSprop等自适应优化算法的核心动机：**利用每个参数的历史梯度信息，动态调整其个性化的步长**。

#### AdaGrad 的缺陷与 RMSprop 的修复

AdaGrad 的"历史梯度平方和"只会累加，永不减少。随着训练进行，累加值越来越大，最终所有参数的个性化步长都趋近于零——模型将无法再学习任何新东西。

RMSprop 不再累加"所有历史"，而是使用**指数移动平均（Exponential Moving Average）**，更看重最近的梯度行为，逐渐"遗忘"久远的梯度。这样分母不会无限增大，模型始终能保持合理的步长。

这一演进精准对应了正文开头提到的缺陷驱动链条：

> **不同参数间梯度尺度差异**（固定步长 → **Adagrad/RMSprop**）

理解了"一刀切"步长的局限和"个性化步长"的解决方案，就能深刻理解从SGD到AdaGrad、再到RMSprop和Adam的演进逻辑——每一步升级都在回答同一个问题：**如何让每个参数的步长恰好匹配它的地形特征？**

### 2.1.2 "最速下降法"名称的再澄清：局部最速 vs 全局最优

既然个性化步长在数学上更优，为什么普通GD还敢叫"最速下降法"？这个名称是否名不副实？

#### 名称来源：数学上的严格定义

这个名称来源于**数学上的严格定义**，而不是工程实践中的表现。

在最速下降法被提出的数学框架中：
- 它是在**当前点**的**无穷小邻域**内讨论的
- 它回答的问题是：**"从这个点出发，哪个方向让函数值下降最快？"**

答案就是：**负梯度方向**。

证明如下：

方向导数 $D_{\boldsymbol u} f(\boldsymbol x_0)$ 表示沿单位方向 $\boldsymbol u$ 的瞬时变化率：

$$
D_{\boldsymbol u} f(\boldsymbol x_0) = \nabla f(\boldsymbol x_0)^\top \boldsymbol u = \|\nabla f\| \cdot \cos \varphi
$$

其中 $\varphi$ 是梯度方向与 $\boldsymbol u$ 的夹角。

- 当 $\varphi = 0°$（沿梯度方向）：$D_{\boldsymbol u} f = +\|\nabla f\|$，上升最快
- 当 $\varphi = 180°$（沿负梯度方向）：$D_{\boldsymbol u} f = -\|\nabla f\|$，**下降最快**

在所有方向中，负梯度方向使方向导数取**全局最小值** $-\|\nabla f\|$。

**所以，"最速下降法"这个名称是从"方向导数"的意义上定义的，是数学上的严格结论。**

#### "最速"在工程中为什么不感觉"最速"？

| 层面 | "最速"的含义 | 成立条件 |
|:---|:---|:---|
| **数学定义** | 方向导数最小（瞬时下降率最大） | $\eta \to 0$（无穷小步长） |
| **工程实践** | 有限步长下的实际下降量最大 | $\eta > 0$ 时**不成立** |

工程中用的步长 $\eta$ 是有限的、非零的。一旦步长不是无穷小：

$$
L(\theta_t - \eta \nabla f) = L(\theta_t) - \eta \|\nabla f\|^2 + O(\eta^2)
$$

**高阶项 $O(\eta^2)$ 出现了！**

- 当 $\eta$ 很小时，$O(\eta^2)$ 可忽略，负梯度方向确实接近最优
- 当 $\eta$ 增大时，$O(\eta^2)$ 的影响不可忽视，负梯度方向可能**远非最优**

#### 形象类比

想象你站在一个复杂的山地上，蒙着眼睛：

- **"最速下降法"** 告诉你：**此时此刻**，脚下最陡的方向是哪个
- 它可以让你准确地朝那个方向迈出**一小步**
- 但如果你迈出**一大步**，你可能已经越过了山脊，甚至掉进了不同的山谷

**"最速"是局部的、瞬时的，而不是全局的、有限步的。**

#### 总结：一张图说清楚

```
"最速下降法"这个名称的含义层次：

数学层面（名称来源）
    ↓
方向导数取最小值 → 负梯度方向是瞬时下降最快的方向
    ↓
严格成立条件：步长 η → 0（无穷小）
    ↓
工程层面（实际使用）
    ↓
步长 η > 0（有限）→ 高阶项 O(η²) 不可忽略
    ↓
负梯度方向不再保证"最速"
    ↓
需要动量、自适应步长等修正
```

**一句话总结**：普通GD被称为"最速下降法"，是因为它在**无穷小步长的极限下**确实是"最速"的，这是从方向导数定义中严格证明的数学结论。但在有限步长的工程实践中，这个"最速"只是**局部的、瞬时的**，而非全局最优——这正是后续所有优化算法（Momentum、AdaGrad、Adam等）存在的根本原因。

## 3 随机梯度下降（SGD）与全量梯度下降（FGD）的二重差异

在深度学习训练中，SGD和FGD的差异常被简单归结为"梯度噪声"。但经过前面的分析可以发现，二者差异远比这更深刻——实际上来源于两个**相互独立、可以解耦**的机制。这一问题在附录F中有完整推导，本节仅做概述性说明。

> **符号约定**：FGD = Full‑Gradient Descent 全量梯度下降；SGD = Stochastic Gradient Descent，逐样本随机梯度下降。
> $W_s$：epoch起始参数；$W_e$：epoch结束参数；上标区分两套迭代序列。

### 3.1 第一重：时序迭代带来的确定性差异
该效应属于确定性机制，和随机无关：即使关闭shuffle，使用固定不变的样本顺序遍历，SGD与FGD的epoch输出依然会不一样。

**FGD迭代**

FGD在整个epoch内参数保持不变，全部样本梯度都在起始参数$W_s$处计算，一次性完成更新：
$$
W_e^{FGD}=W_s-\eta\sum_{i=1}^n \nabla L\big(y_i,f(W_s,X_i)\big)
$$
所有梯度求值点完全相同，求和满足加法交换律；**样本先后顺序不会影响FGD输出结果**。


**SGD迭代**（无shuffle、固定顺序）

SGD核心行为：每处理完一个样本，立刻更新参数；下一个样本的梯度，在**已经更新后的新参数**上重新计算。

设epoch起始参数为 $W_s$：

- 第1个样本，梯度在起始点$W_s$计算，得到第一步更新：
$$W_1^{SGD}=W_s-\eta \nabla L\big(y_1,f(W_s,X_1)\big)$$

- 第2个样本，梯度在更新后的 $W_1^{SGD}$ 上计算：
$$W_2^{SGD}=W_1^{SGD}-\eta \nabla L\big(y_2,f(W_1^{SGD},X_2)\big)$$

- 第$i$步通用迭代：
$$W_{i}^{SGD}=W_{i-1}^{SGD}-\eta \nabla L\big(y_i,f(W_{i-1}^{SGD},X_i)\big)$$

遍历全部$n$个样本，epoch结束得到终点参数 $W_e^{SGD}$：
$$
\boldsymbol{W_e^{SGD}=W_s - \eta \sum_{i=1}^{n} \nabla L\big(y_i,f(W_{i-1}^{SGD},X_i)\big)}
$$

#### 关键核心结论
1. SGD求和中每一项梯度求值点 $W_{i-1}^{SGD}$ 互不相同，每一步梯度都依赖前面样本带来的参数改变；
2. 加法交换律失效：调换样本顺序，每一项梯度的求值点全部发生改变，最终 $W_e^{SGD}$ 必然改变；
3. 线性模型特例：当梯度与参数本身无关，该确定性差异消失，一轮SGD等价FGD。

### 3.2 第二重：乱序（shuffle）采样带来的随机差异
随机差异产生的根源：**固定同一个模型参数$W$不变时，不同样本计算出来的样本损失不一样，对应的单样本梯度自然各不相同**。

FGD把全部样本损失求和取平均，样本个体差异互相抵消，输出唯一确定的更新，不存在该随机效应。

SGD逐样本更新，每一步只用某一个样本梯度近似全集梯度。`shuffle=True`只是对数据集做无放回随机重排，样本本身不变，只改变出场次序。

 **实例看差异如何发生（$W$全程固定不动）**
数据集两个样本A、B，模型参数$W$完全固定。
- 当前参数下样本A损失大，对应单样本梯度 $g_A$幅值大；
- 当前参数下样本B损失小，对应单样本梯度 $g_B$幅值小；

1）shuffle序列[A,B]，第一步用$g_A$更新：
$$W' = W - \eta \cdot g_A$$
2）另一次shuffle得到序列[B,A]，第一步用$g_B$更新：
$$W'' = W - \eta \cdot g_B$$

$g_A \neq g_B \implies W' \neq W''$。
> 关键点：初始输入参数完全一致；差异**不是迭代修改W造成**，仅仅shuffle随机选出不同样本；样本各自损失不同，单样本梯度不同，第一步更新量就出现差别，这就是乱序采样带来的随机差异。

该初始差别会向后传递放大：第一步得到不同参数，后续样本的损失梯度基于新参数计算，整条迭代轨迹产生随机偏移。

统计意义上，单样本梯度是全集梯度的无偏估计；但样本梯度之间存在离散，带来估计方差，也就是常说的梯度噪声。

> 📌 和第一重确定性差异严格区分
> - 第二重随机差异：起始$W$固定不变；根源在于样本之间损失、梯度的个体差异，shuffle随机挑选样本，造成更新量随机不同。
> - 第一重确定性差异：不需要随机；epoch内部每一步迭代直接修改$W^{SGD}$；后续损失梯度在已经改变的参数上计算。



真实训练开启shuffle时两种效应耦合在一起：shuffle引入随机梯度估计差异；紧接着每一步参数修改，触发时序迭代的确定性差异，共同决定迭代路径。

### 四种场景，二重差异组合对照表

| 训练场景 | 第一重：时序迭代确定性差异 | 第二重：乱序采样随机差异 |
|---|---|---|
| FGD全量梯度下降 | ❌ 无 | ❌ 无 |
| 确定性SGD（无shuffle、固定顺序） | ✅ 存在 | ❌ 无（本次实验场景） |
| 随机性SGD（有shuffle） | ✅ 存在 |  ✅ 存在 |

#### 实验（验证第一重确定性差异）

非线性模型 $\hat y = w^2 x$，MSE损失：
$$
L=\frac12(y-\hat y)^2
\quad\Rightarrow\quad
\frac{\partial L}{\partial w}=-(y-w^2 x)\cdot 2wx
$$

样本：$point_1=(x=1.0,y=2.0),\ point_2=(x=2.0,y=3.0)$
- epoch起始参数 $w_s=1.0$
- 学习率 $\eta=0.05$
- 使用两套固定顺序，关闭随机，对比epoch终点输出 $w_e^{SGD}$。

In [ ]:
import numpy as np

def grad_L(w, x, y):
    """
    计算单样本损失对参数w的梯度
    模型：yhat = w² * x
    损失：MSE  L = 1/2*(y‑yhat)²
    解析求导：dL/dw = -(y − w²·x) · 2·w·x

    Parameters
    ----------
    w : float
        当前模型参数
    x : float
        样本输入特征
    y : float
        样本标签真值

    Returns
    -------
    grad : float
        该样本下损失对w的梯度
    """
    yhat = (w ** 2) * x   # 模型前向计算预测值
    grad = -(y - yhat) * (2 * w * x)   # MSE损失解析梯度
    return grad


def deterministic_sgd_epoch(w_s, eta, data_order):
    """
    【确定性SGD】完整遍历一轮epoch，无shuffle，样本顺序固定
    核心行为：每处理1个样本，立刻更新参数；下一个样本梯度基于更新后的w计算。
    对应公式：$W_{i}^{SGD}=W_{i-1}^{SGD}-\eta \\nabla L\big(y_i,f(W_{i-1}^{SGD},X_i)\big)$

    Parameters
    ----------
    w_s : float
        epoch起始参数 start，对应符号 $W_s = W_1^{SGD}$
    eta : float
        学习率
    data_order : list[tuple]
        人为指定的样本遍历顺序，关闭随机shuffle

    Returns
    -------
    w : float
        epoch迭代结束后的终点参数，对应 $w_e^{SGD}$
    trace : list[float]
        参数完整轨迹，trace[0]=初始值，每完成1个样本追加1次更新后w
    """
    w = w_s                  # 初始化：将epoch起始参数赋值给迭代变量
    trace = [w]              # 轨迹列表，保存每一步参数；第一个元素为epoch起点
    for (x, y) in data_order:
        g = grad_L(w, x, y)  # 用当前最新w，计算该样本的单样本梯度
        w = w - eta * g      # SGD更新：使用单样本梯度做参数迭代
        trace.append(w)      # 将更新完成后的参数存入轨迹
    return w, trace


def full_gd_step(w_s, eta, data):
    """
    FGD全量梯度下降：一个epoch只执行1次参数更新
    全部n个样本梯度统一在**同一个起始参数w_s**处计算，求和之后一次性更新。
    对应公式：$W_2^{FGD}=W_s-\eta\sum_{i=1}^n \nabla L\big(y_i,f(W_s,X_i)\big)$

    Parameters
    ----------
    w_s : float
        epoch起始参数，所有样本梯度都在该点求值
    eta : float
        学习率
    data : list[tuple]
        全部数据集样本，顺序不影响FGD结果（加法交换律）

    Returns
    -------
    w_fgd : float
        FGD完成唯一一次更新之后输出的参数 $W_2^{FGD}$
    """
    total_grad = 0.0                     # 累加全部样本的梯度
    for (x, y) in data:
        total_grad += grad_L(w_s, x, y)  # 全部样本梯度都在w_s处计算
    w_fgd = w_s - eta * total_grad       # 汇总全部梯度，仅做一次参数更新
    return w_fgd


# ----------------------运行实验-------------------------
# 构造2个测试样本：(特征x，标签y)
point1 = (1.0, 2.0)
point2 = (2.0, 3.0)

w_s = 1.0      # epoch起始参数 W_s
eta = 0.05     # 学习率

# 实验：两套完全相反的固定样本顺序，关闭随机shuffle，观察确定性差异
# 顺序A：先point1，后point2
w_e_A, trace_A = deterministic_sgd_epoch(w_s, eta, [point1, point2])
# 顺序B：先point2，后point1
w_e_B, trace_B = deterministic_sgd_epoch(w_s, eta, [point2, point1])
# FGD全量梯度下降作为参照基准
w_fgd_out = full_gd_step(w_s, eta, [point1, point2])

# 打印输出，与推导公式符号一一对应
print(f"epoch起始参数 w_s = {w_s:.6f}")
print(f"\n顺序 [point1, point2]，迭代轨迹：{[round(v, 6) for v in trace_A]}")
print(f"epoch终点 w_e^{{SGD}}(A) = {w_e_A:.6f}")
print(f"\n顺序 [point2, point1]，迭代轨迹：{[round(v, 6) for v in trace_B]}")
print(f"epoch终点 w_e^{{SGD}}(B) = {w_e_B:.6f}")
print(f"\nFGD输出 w_2^{{FGD}} = {w_fgd_out:.6f}")
# 判断：无随机，仅调换样本顺序，epoch终点参数是否不同，验证第一重确定性差异
print(f"\nw_e_A != w_e_B ? {not np.isclose(w_e_A, w_e_B)}")

#### 实验结果解读



1. 无任何随机，仅仅调换样本顺序，$w_e^{SGD}$就发生变化，验证第一重**时序迭代确定性差异真实存在**；
2. 两套SGD终点，均不等于FGD输出 $w_2^{FGD}$；即使消除全部随机噪声，SGD与FGD输出依然不一致。


开启shuffle的真实SGD，两套效应同时起作用：
- 时序迭代确定性差异：epoch内部参数不断被改写，样本顺序改变迭代路径；
- 乱序采样随机差异：样本损失梯度本身离散，shuffle引入梯度估计噪声，帮助逃离局部极小，提升泛化。
> 注意：梯度噪声是**盲目扰动，属于副作用**，没有理论保证一定逃离局部极小；只是存在概率跳出部分局部极小，既不能保证一定跳出，也不能保证跳向更优的参数区域。实践中观察到泛化能力提升是统计层面的经验现象。

### Mini‑batch场景下的二重差异
Mini‑batch梯度下降同时包含两套机制：
1. **第二重随机差异（batch内部）**：同一参数下，不同batch样本构成不同，子集损失分布不同，batch梯度估计结果不同；batch越小，样本越少，个体差异难以抵消，随机波动越大。
2. **第一重确定性差异（batch之间）**：每完成一个batch就更新参数，下一批样本的损失、梯度在新参数上求解，时序迭代效应持续生效。

batch增大，样本个体差异被平均，随机差异被抑制；只要batch不等于全集样本，**时序迭代带来的确定性差异始终存在**。

## 常见误区

### 误区一：SGD和FGD的区别只有梯度噪声

很多说法认为SGD和FGD区别只有梯度噪声。
这个认知不完整：**关闭shuffle，彻底消除随机噪声，固定顺序逐样本SGD，输出$W_e^{SGD}$依然不等于FGD输出$W_2^{FGD}$**。

SGD与FGD之间总差异 = 时序迭代带来的确定性差异 + 乱序采样带来的随机差异；二者相互独立，共同起作用。

---

### 误区二：GD会"受困于鞍点"——驻点问题是GD的重大困境

一个常见的说法是：梯度下降法在非凸函数上会"受困于鞍点"，这是深度学习训练的主要障碍之一。

**仿真演示**：

选取具有多个局部极值和鞍点的震荡函数 $f(x) = x\sin(x^2) + 1$，从5个不同初始点出发执行梯度下降法，观察收敛行为。

In [ ]:
# ===== 导入必要的库 =====
import numpy as np                      # 数值计算库
import plotly.graph_objects as go      # 交互式可视化库
from scipy.optimize import minimize_scalar  # 用于辅助寻找极值点（本实验中未直接使用）

# ========= 非凸函数定义 =========
def f_nonconvex(x):
    """
    非凸函数：f(x) = x * sin(x^2) + 1
    该函数在定义域内存在多个局部极大值、局部极小值和鞍点
    """
    return x * np.sin(x**2) + 1

def df_nonconvex(x):
    """
    非凸函数的一阶导数：
    f'(x) = sin(x^2) + 2 * x^2 * cos(x^2)
    使用链式法则求导：d/dx [x * sin(x^2)] = sin(x^2) + x * cos(x^2) * 2x
    """
    return np.sin(x**2) + 2 * x**2 * np.cos(x**2)

# ========= 定义域边界 =========
DOMAIN_L = -2.5    # 定义域左边界
DOMAIN_R = 2.5     # 定义域右边界

# ========= 多起点实验配置 =========
initial_points = [-1.5, -0.5, 0.5, 2.0, 2.4]  # 五个不同初始点
eta = 0.02                                    # 步长（取较小值以保证在震荡函数上稳定）
epsilon = 0.001                                # 梯度收敛阈值
max_iter = 80                                 # 最大迭代次数

# ========= 生成函数曲线 =========
x_curve = np.linspace(-2.6, 2.6, 3000)
y_curve = f_nonconvex(x_curve)

# ========= 创建图形 =========
fig = go.Figure()

# 绘制非凸函数曲线（黑色粗线）
fig.add_trace(go.Scatter(
    x=x_curve,
    y=y_curve,
    mode="lines",
    line=dict(color="black", width=2.5),
    showlegend=False
))

# 绘制定义域边界（灰色虚线）
fig.add_vline(x=DOMAIN_L, line_dash="dot", line_color="gray", opacity=0.6)
fig.add_vline(x=DOMAIN_R, line_dash="dot", line_color="gray", opacity=0.6)

# ========= 颜色方案 =========
colors = ['tomato', 'limegreen', 'royalblue', 'orange', 'mediumpurple']

# 存储收敛结果
convergence_results = []

# ========= 对每个初始点执行梯度下降法 =========
for init_idx, x0 in enumerate(initial_points):
    color = colors[init_idx % len(colors)]

    xs = [x0]                                 # 记录迭代路径
    ys = [f_nonconvex(x0)]
    x_current = x0

    for k in range(max_iter):
        grad = df_nonconvex(x_current)        # 计算当前梯度
        x_next = x_current - eta * grad       # 梯度下降更新
        x_next = np.clip(x_next, DOMAIN_L + 1e-6, DOMAIN_R - 1e-6)  # 限制在定义域内

        xs.append(x_next)                     # 记录新位置
        ys.append(f_nonconvex(x_next))        # 记录新位置函数值

        if abs(grad) < epsilon:               # 梯度足够小则停止
            break
        x_current = x_next                    # 更新当前参数

    # 保存收敛结果
    convergence_results.append({
        'start': x0,
        'end': xs[-1],
        'loss': ys[-1],
        'steps': len(xs) - 1,
        'color': color
    })

    # 绘制起始点（六边形）
    fig.add_trace(go.Scatter(
        x=[xs[0]],
        y=[ys[0]],
        mode="markers",
        marker=dict(
            color=color,
            size=12,
            symbol='hexagon',
            line=dict(width=2, color='white')
        ),
        showlegend=False
    ))

    # 绘制迭代路径（彩色虚线连接线）
    for i in range(len(xs)-1):
        fig.add_trace(go.Scatter(
            x=[xs[i], xs[i+1]],
            y=[ys[i], ys[i+1]],
            mode="lines",
            line=dict(color=color, width=1.2, dash='dot'),
            showlegend=False
        ))

    # 绘制中间迭代点（小圆点）
    fig.add_trace(go.Scatter(
        x=xs[1:-1],
        y=ys[1:-1],
        mode="markers",
        marker=dict(color=color, size=4),
        showlegend=False
    ))

    # 绘制收敛终点（星形）
    fig.add_trace(go.Scatter(
        x=[xs[-1]],
        y=[ys[-1]],
        mode="markers",
        marker=dict(
            color=color,
            size=14,
            symbol='star',
            line=dict(width=2, color='white')
        ),
        showlegend=False
    ))

# ========= 图例说明 =========
# 添加图例条目（使用空数据点，仅用于显示图例）
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode="markers",
    marker=dict(symbol="hexagon", size=12, color="gray", line=dict(width=2, color="white")),
    name="迭代起点"
))

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode="markers",
    marker=dict(symbol="star", size=14, color="gray", line=dict(width=2, color="white")),
    name="迭代收敛点（颜色与迭代路径颜色相同）"
))

# 绘制 y=0 参考线（浅灰色虚线）
fig.add_hline(y=0, line_dash="dash", line_color="lightgray", opacity=0.5)

# ========= 设置图形布局 =========
fig.update_layout(
    title=None,
    xaxis_title=None,
    yaxis_title=None,
    template="plotly_white",
    width=1200,
    height=700,
    xaxis=dict(range=[-2.6, 2.6]),
    hovermode="x unified",
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.98,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="lightgray",
        borderwidth=1,
        font=dict(size=10)
    ),
    annotations=[]
)

fig.show()

# ========= 打印收敛结果汇总 =========
print("\n" + "=" * 70)
print("【梯度下降法收敛结果汇总 — 定义域 (-2.5, 2.5) 开区间】")
print("=" * 70)

for res in convergence_results:
    print(f"起点x={res['start']:.1f} → 收敛点 x = {res['end']:.4f}, f(x) = {res['loss']:.4f}")

print("=" * 70)
print("💡 关键观察：起点x=0.5 收敛到 x≈0.1458, f(x)≈1.0031")
print("   该点梯度 ≈ 0，但并非局部极小值——这是一个典型的鞍点！")

**从仿真中直观看到**：起点 $x=0.5$ 收敛到 $x \approx 0.1458$，该点梯度 $f'(x) \approx 0$，但函数值 $f(x) \approx 1.0031$ 并非该区域的最小值——**这是一个鞍点**。在确定性GD、无噪声、一维非凸函数的设定下，梯度下降法确实会在鞍点处停滞。

**为什么这是误区？**

上述仿真成立的前提是**确定性GD + 精确梯度 + 一维**。但深度学习的真实训练条件与这个设定有本质不同：

| 条件 | 仿真中的设定 | 深度学习中的真实情况 |
|:---|:---|:---|
| **梯度估计** | 精确梯度，无噪声 | Mini-batch随机估计，持续扰动 |
| **维度** | 一维 | 百万到千亿维 |
| **精确停在驻点的概率** | 取决于初始值，可能发生 | 参数从连续分布中采样，概率为零 |
| **更新机制** | 纯GD，无历史记忆 | Momentum/Adam累积历史速度 |

**准确的说法**：

1. **理论上**：鞍点是GD的数学困境——梯度为零时更新停止，这是严格成立的。在确定性、无噪声、低维场景下，这个困境真实存在。

2. **工程上**：深度学习的真实训练条件使鞍点困境的实际影响被显著削弱：
   - Mini-batch SGD的随机梯度噪声提供了持续扰动，参数几乎不可能精确停在梯度为零的点；
   - 动量累积的历史速度在梯度接近零时仍然存在，可以"冲过"鞍点区域；
   - 高维空间中，即使某些维度梯度为零，其他维度的梯度噪声也能推动参数继续移动。

3. **因此**：把鞍点列为GD的"根本困境"是一种**过度强调**。GD在深度学习中真正的主要困境是**步长敏感**和**方向短视**（见1.4节），而非鞍点。

> 📌 **一句话澄清**：鞍点问题在确定性GD + 低维非凸函数中确实存在，但在深度学习的工程现实中，随机噪声、高维特性与动量机制共同作用，使其实际影响远小于步长敏感和方向短视。

In [ ]:
import numpy as np

def grad_L(w, x, y):
    """
    计算单样本损失对参数w的梯度
    模型：yhat = w² * x
    损失：MSE  L = 1/2*(y‑yhat)²
    解析求导：dL/dw = -(y − w²·x) · 2·w·x
    """
    yhat = (w ** 2) * x
    grad = -(y - yhat) * (2 * w * x)
    return grad


def deterministic_sgd_epoch(w_s, eta, data_order):
    """
    【确定性SGD】完整遍历一轮epoch，无shuffle，样本顺序固定
    核心行为：每处理1个样本，立刻更新参数；下一个样本梯度基于更新后的w计算。
    """
    w = w_s
    trace = [w]
    for (x, y) in data_order:
        g = grad_L(w, x, y)
        w = w - eta * g
        trace.append(w)
    return w, trace


def full_gd_step(w_s, eta, data):
    """
    FGD全量梯度下降：一个epoch只执行1次参数更新
    全部n个样本梯度统一在**同一个起始参数w_s**处计算，求和之后一次性更新。
    """
    total_grad = 0.0
    for (x, y) in data:
        total_grad += grad_L(w_s, x, y)
    w_fgd = w_s - eta * total_grad
    return w_fgd


# ----------------------运行实验-------------------------
point1 = (1.0, 2.0)
point2 = (2.0, 3.0)

w_s = 1.0
eta = 0.05

# 顺序A：先point1，后point2
w_e_A, trace_A = deterministic_sgd_epoch(w_s, eta, [point1, point2])
# 顺序B：先point2，后point1
w_e_B, trace_B = deterministic_sgd_epoch(w_s, eta, [point2, point1])
# FGD全量梯度下降作为参照基准
w_fgd_out = full_gd_step(w_s, eta, [point1, point2])

print(f"epoch起始参数 w_s = {w_s:.6f}")
print(f"\n顺序 [point1, point2]，迭代轨迹：{[round(v, 6) for v in trace_A]}")
print(f"epoch终点 w_e^SGD(A) = {w_e_A:.6f}")
print(f"\n顺序 [point2, point1]，迭代轨迹：{[round(v, 6) for v in trace_B]}")
print(f"epoch终点 w_e^SGD(B) = {w_e_B:.6f}")
print(f"\nFGD输出 w_e^FGD = {w_fgd_out:.6f}")
print(f"\nw_e_A != w_e_B ? {not np.isclose(w_e_A, w_e_B)}")

## 附录A：GD迭代路径垂直于等高线的几何证明



> **正文引用位置**：在"先睹为快"算例的观察总结第2点中，正文指出"梯度方向垂直于等高线"是"梯度下降法的几何本质"，并标注"详见附录A"。

---

### A-1 问题的核心

在正文的二元对称凸二次函数算例中，我们从等高线图上直观看到：梯度下降的迭代路径与等高线相交时，**总是以90°角穿过**。

这并非巧合，而是由梯度的几何本质决定的。本附录从两个层面给出证明：

1. **梯度的几何意义**：梯度向量是等高线的法向量
2. **方向导数的极值条件**：负梯度方向是下降最快的方向

---

### A-2 证明一：梯度是等高线的法向量（几何证明）

设目标函数为 $f(x, y)$，其等高线定义为 $f(x, y) = C$（$C$ 为常数）。

对等高线方程两边同时求全微分：

$$
df = \frac{\partial f}{\partial x} dx + \frac{\partial f}{\partial y} dy = 0
$$

将上式写成向量的点积形式：

$$
\left( \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y} \right) \cdot (dx, dy) = 0
$$

其中：
- 左边的向量 $\left( \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y} \right)$ **正是梯度向量** $\nabla f$
- 右边的向量 $(dx, dy)$ 是**等高线的切线方向**（因为等高线上函数值不变，即 $df = 0$）

**两个向量点积为 0**，意味着梯度向量与等高线的切线方向**相互垂直**。

因此：

$$
\boxed{\nabla f \perp \text{等高线切线}}
$$

既然梯度垂直于切线，那么梯度就是等高线的**法线方向**。

梯度下降法沿着 $\Delta\theta = -\eta \nabla f$ 方向更新——这就是沿着等高线的**法线方向**前进。所以迭代路径必然以90°角穿过等高线。

---

### A-3 证明二：方向导数极值（"最速"的数学根源）

方向导数 $D_{\boldsymbol u} f(\boldsymbol x_0)$ 表示沿单位方向 $\boldsymbol u$ 的**瞬时变化率**：

$$
D_{\boldsymbol u} f(\boldsymbol x_0) = \nabla f(\boldsymbol x_0)^\top \boldsymbol u
$$

由于 $\boldsymbol u$ 是单位向量，根据向量点积的几何定义：

$$
D_{\boldsymbol u} f(\boldsymbol x_0) = \|\nabla f\| \cdot \cos \varphi
$$

其中 $\varphi$ 是梯度方向与 $\boldsymbol u$ 的夹角。

**分析极值**：

| 方向 | $\varphi$ | 方向导数 | 含义 |
|:---|:---|:---|:---|
| 沿梯度方向 | $\varphi = 0°$ | $D_{\boldsymbol u} f = +\|\nabla f\|$ | 上升最快 |
| 沿负梯度方向 | $\varphi = 180°$ | $D_{\boldsymbol u} f = -\|\nabla f\|$ | **下降最快** |
| 沿等高线方向 | $\varphi = 90°$ | $D_{\boldsymbol u} f = 0$ | 函数值不变 |

**关键结论**：负梯度方向在所有方向中使方向导数取**全局最小值** $-\|\nabla f\|$。

而方向导数 $D_{\boldsymbol u} f = 0$ 的方向正是等高线的切向。负梯度方向与等高线切向的夹角为 $180° - 90° = 90°$，再次印证了**迭代路径垂直于等高线**。

---

### A-4 两个"反直觉"的细节澄清

在实际的机器学习应用中，你可能会觉得"看到的GD路径好像并不完全垂直"，这通常源于以下两个因素：

| 因素 | 说明 |
|:---|:---|
| **等高线图是俯视图** | 在三维曲面上，GD的每一步都是沿着最陡峭的坡面直线向下。但在俯视的等高线平面图上，这条路径看起来是**沿着法线方向**切过每一条等高线的 |
| **有限步长的影响** | 只有**步长趋于无限小**（即微分意义上的"最速下降"）时，路径才严格垂直于等高线。如果步长较大，GD会沿着当前点的法线方向走一大步，落到新的等高线上后，方向会**重新计算**，变成新位置的法线方向。因此宏观上它是一条折线，但**在每一个迭代点上**，其出发方向都是严格垂直于该点等高线的 |

---

### A-5 速查总结

| 问题 | 答案 |
|:---|:---|
| **梯度为什么垂直于等高线？** | 对 $f(x,y)=C$ 全微分得 $\nabla f \cdot (dx,dy)=0$，$(dx,dy)$ 是切线方向，故梯度⊥切线 |
| **负梯度方向如何？** | 方向导数 $D_{\boldsymbol u}f = \|\nabla f\|\cos\varphi$，$\varphi=180°$ 时取最小值，即下降最快 |
| **有限步长下还垂直吗？** | 只在**当前点**的出发方向上严格垂直；步长有限时，整体路径是折线，但在每个迭代点处仍是垂直出发 |
| **和"最速下降法"的关系？** | 垂直于等高线 = 沿法线方向 = 瞬时下降最快的方向——这正是"最速下降法"名称的几何解释 |

---

**附录A一句话总结**：梯度向量是等高线的法向量，负梯度方向使方向导数取全局最小值——这两个独立证明共同指向同一个几何事实：**GD的迭代路径以90°角穿过等高线，这是梯度下降法"沿最陡方向下山"这一核心思想的直接几何表达**。




## 附录B：梯度（或次梯度）的"存在"与"可计算"

本附录解释1.1.2节核心适用条件中 **"梯度（或次梯度）必须存在且可计算"** 这句话的具体含义，拆解"存在"与"可计算"两个维度的独立要求。

---

### B-1 拆解两层含义

#### B-1.1 存在性：梯度在数学上必须定义良好

| 函数在该点状态 | 梯度是否存在 | 能否用标准GD |
|:---|:---|:---|
| **可导** | ✅ 梯度存在且唯一 | ✅ 可以使用 |
| **不可导但凸** | ❌ 梯度不存在，但**次梯度存在** | ⚠️ 可用次梯度法（收敛慢） |
| **不可导且非凸** | ❌ 梯度不存在，次梯度也**没有定义** | ❌ 标准GD完全失效 |

**"存在"的含义**：

- 函数在某点的梯度是一个**确定的向量**，它必须存在；
- 如果梯度不存在，但函数是凸函数，则可以用**次梯度**替代（至少有一个次梯度）；
- 如果函数既不可导又不是凸函数，则**没有梯度也没有次梯度**，GD无法起步。

#### B-1.2 可计算性：工程上能算出具体数值

即使梯度在数学上存在，如果**无法获得具体数值**，GD仍然无法执行：

| 情况 | 能否计算梯度 | 能否用GD |
|:---|:---|:---|
| 解析表达式已知（如$f(x)=x^2$） | ✅ 手算或符号求导 | ✅ |
| 自动微分支持（神经网络） | ✅ 框架自动计算 | ✅ |
| 黑盒函数（无解析形式） | ❌ 无法求导 | ❌ |
| 手动推导复杂但可行 | ✅ 付出人力可算 | ✅（但易出错） |

---

### B-2 为什么"存在但不可计算"是真实的困境？

"梯度在数学上存在，但无法实际计算出来"是真实存在的，尤其在数学和工程交界处。

#### B-2.1 隐函数/隐式梯度

**场景**：目标函数本身没有显式表达式，梯度虽然理论上存在，但无法写成解析式。

**例**：某些物理仿真或大规模数值模拟中，损失是仿真结果，虽然输入到输出是连续可微的，但**导数无法解析推导**，只能通过有限差分近似——而高维参数空间下有限差分计算量爆炸，几乎不可行。

#### B-2.2 极端复杂的复合函数

**场景**：函数包含复杂子模块（如求解器、迭代算法），理论上可导，但链式法则展开后**表达式天文数字般庞大**。

**例**：深度均衡模型（DEQ）的梯度涉及求解逆矩阵，数学上存在，但直接计算需要 $O(n^3)$ 矩阵求逆，对于百万参数模型完全不可行（虽然存在，但"不可计算"）。

#### B-2.3 导数存在但数值不稳定

**场景**：梯度存在，但数值计算时发生灾难性抵消或溢出，导致计算出的结果不可信。

**例**：$f(x) = \log(1+e^x)$ 在 $x$ 很大时，理论梯度 $\sigma(x)$ 趋近于1，但直接计算 $e^x$ 会溢出（`inf`），程序崩溃——这个场景中梯度"存在"且理论上可算，但**实际硬件/软件环境中无法稳定计算**。

---

### B-3 深度学习中遇到的情况

| 场景 | 梯度是否存在？ | 能否计算？ | 如何应对？ |
|:---|:---|:---|
| 标准神经网络 + 自动微分 | ✅ 存在 | ✅ 可计算 | 直接使用框架 |
| DEQ（隐式微分） | ✅ 存在 | ❌ 直接算不可行（矩阵逆 $O(n^3)$） | 用不动点迭代近似求解（Neumann级数） |
| ReLU在0点 | ❌ 梯度不存在 | ✅ 次梯度可计算 | 框架取默认值（0或0.5） |
| 黑盒模拟器 | ✅ 可能存在 | ❌ 无解析梯度 | 用无梯度优化（贝叶斯优化/进化策略） |

---

### B-4 哲学视角：存在与可计算的分离

"理论上存在不等于工程上能拿到"——这一点与**图灵停机问题**有内在相似性：并非所有数学上定义良好的对象都能在有限时间内构造出来。

---

### B-5 速查总结

| 条件 | 含义 | 为什么独立？ |
|:---|:---|:---|
| **存在** | 数学上有定义 | 缺了它GD无法形式化推导 |
| **可计算** | 实践中能算出数值 | 存在但算不出 → GD仍然无法执行 |

---

**附录B一句话总结**："梯度（或次梯度）必须存在且可计算"把两个独立条件并列列出——**存在**确保数学上有定义，**可计算**确保工程上能拿到具体数值。存在但算不出，GD仍然无法执行。




## 附录C：凸函数与凸集

本附录解释什么是凸函数和凸集，以及两者之间的内在联系。这两个概念是理解梯度下降法在凸优化中具有全局收敛保证的理论基石。

---

### C-1 凸集（Convex Set）

**定义**：在一个集合中，任取两个点，连接它们的线段**完全包含**在该集合内，那么这个集合就是凸集。

**直观理解**：
- **凸集**：像一颗**实心圆球**、一块**正方形木板**、或者一个**实心立方体**。
- **非凸集**：像**甜甜圈（圆环）**、**月牙形**、或者**空心球壳**。

**数学表达**：

设集合 $C$，如果对于 $C$ 中的任意两点 $x_1$ 和 $x_2$，以及任意 $\lambda \in [0, 1]$，都有：
$$
\lambda x_1 + (1-\lambda)x_2 \in C
$$
那么 $C$ 就是凸集。

**常见凸集**：整个空间、直线、射线、圆形/椭圆形内部、超平面等。

---

### C-2 凸函数（Convex Function）

**定义**：函数图像上任意两点之间的连线，位于函数图像的上方（或重合）。

**几何理解**（"碗形"）：

想象一个碗（开口向上），比如函数 $y = x^2$。在图像上取任意两点，用一条直线把它们连起来，这条直线一定在碗的上方。这种形状就是凸的。

- **形象记忆**：像笑脸 😊 一样的曲线就是凸函数（开口朝上）。

> **注意术语差异**：在国内高等数学教材中，有时会把开口向上的称为"凹函数"，而把开口向下的称为"凸函数"。但在**现代运筹学和机器学习领域（国际通用标准）**中，我们统一使用"凸函数 = 开口向上（笑脸）"。本文采用这个主流定义。

**数学定义**：

设函数 $f(x)$ 的定义域是凸集，对于定义域内的任意两点 $x_1$ 和 $x_2$，以及任意 $\lambda \in [0, 1]$，满足：
$$
f(\lambda x_1 + (1-\lambda)x_2) \le \lambda f(x_1) + (1-\lambda)f(x_2)
$$

**如何快速判断（用二阶导数）**：

对于一元函数，如果 $f''(x) \ge 0$ 恒成立，那么函数是凸的。
- $y = x^2$，二阶导数为 2 > 0，是凸函数。
- $y = e^x$，二阶导数还是 $e^x > 0$，也是凸函数。
- $y = -x^2$，二阶导数为 -2 < 0，是**凹函数**（开口向下，像哭脸 ☹️）。

---

### C-3 凸函数与凸集的关系（"舞台"与"演员"）

凸函数和凸集关系非常密切。凸集是定义凸函数的"舞台"，而凸函数是刻画这个舞台上"地势高低"的工具。

#### 关系一：定义域必须是凸集

凸函数的定义里，有一个隐藏条件：**函数的定义域必须是凸集**。如果定义域不凸，函数就谈不上是凸函数。

#### 关系二：凸函数的"上方图"一定是凸集（核心联系）

有一个非常漂亮的几何等价关系：

> **函数是凸函数 ⇔ 它的"上方图"（epigraph）是凸集。**

- 什么是**上方图**？就是函数图像上方所有点的集合（包括图像本身）。
- 如果一个函数的"碗"是凸的（碗壁向上弯曲），那么碗口以上的整个区域自然是凸的；反之亦然。

**经典对应**：

| 概念 | 数学表达式 | 含义 |
|:---|:---|:---|
| **凸集** | $\lambda x_1 + (1-\lambda)x_2 \in C$ | 两点连线上的点还在集合里。 |
| **凸函数** | $f(\lambda x_1 + (1-\lambda)x_2) \le \lambda f(x_1) + (1-\lambda)f(x_2)$ | 函数在连线上的值，小于等于端点值的连线。 |

**一句话总结**：

**凸集是"场地"（定义域），凸函数是"地形"（高度）**。只有场地是完整无缺（凸）的，地形才能呈现出"碗状"（凸函数）的完美形态。

---

### C-4 为什么凸函数在优化中如此重要？

在人工智能和大数据领域，我们经常要做一件事——**求最小值**（比如让预测误差最小）。

- **凸函数的超级福利**：任何一个局部最小值，都一定是全局最小值。不会存在多个"坑"让你陷入局部最优解。
- 正因为这个特性，像梯度下降法这样的算法在凸函数上表现极佳，能保证找到最优解。
- 在机器学习中，**强凸损失**（如L2正则化线性回归）和**一般凸损失**（如逻辑回归）都享有这种理论保证。

---

### C-5 速查总结

| 问题 | 答案 |
|:---|:---|
| **凸集是什么？** | 集合中任意两点连线仍在该集合内，无凹陷、无孔洞。 |
| **凸函数是什么？** | 函数图像上任意两点连线在图像上方（碗朝上，笑脸）。 |
| **凸函数的判断？** | 一元函数二阶导数 $f''(x) \ge 0$。 |
| **凸函数与凸集的关系？** | 定义域必须是凸集；函数的"上方图"是凸集。 |
| **为什么重要？** | 局部最优 = 全局最优，梯度下降法有收敛保证。 |
| **与次梯度的关系？** | 凸函数在不可导点存在次梯度，这是次梯度法的理论基础（详见附录D）。 |

---

**附录C一句话总结**：凸集是"无凹陷"的集合，凸函数是"碗朝上"的函数。两者通过"上方图为凸集"紧密相连，共同构成了凸优化理论的核心，也是梯度下降法在凸损失上具有全局收敛保证的根源。




## 附录D：次梯度（Subgradient）详解

本附录系统讲解次梯度的定义、数学原理、与凸函数的关系，以及在深度学习工程实践中的真实角色。

---

### D-1 什么是次梯度？

次梯度是**梯度概念的推广**，用于处理**不可导的凸函数**。当函数在某点不可导时，梯度不存在，但**次梯度一定存在**（对凸函数而言）。

**从梯度的局限性说起：**

梯度要求函数在该点**处处可微**——所有方向的导数都存在且相等。但现实中有大量**不可导但凸**的函数，例如：

$$f(x) = |x|$$

在 $x=0$ 处：
- 左导数：$-1$
- 右导数：$+1$
- **梯度不存在**

然而梯度下降法仍然需要某种"方向信息"来更新参数——次梯度应运而生。

---

### D-2 次梯度的严格数学定义

对于**凸函数** $f: \mathbb{R}^n \rightarrow \mathbb{R}$，向量 $g \in \mathbb{R}^n$ 称为 $f$ 在点 $x_0$ 处的**次梯度**，当且仅当对**定义域内所有** $x$，满足：

$$f(x) \geq f(x_0) + g^\top (x - x_0)$$

**几何含义**：次梯度 $g$ 定义了**一条始终在函数图像下方的支撑线（超平面）**，它触及 $f$ 于点 $(x_0, f(x_0))$。

---

### D-3 从"唯一切线"到"切线束"

| | 梯度 (可导点) | 次梯度 (不可导点) |
|:---|:---|:---|
| **数量** | 唯一 | 可能多个，构成一个集合 |
| **几何意义** | 唯一的支撑超平面 | 所有可能的支撑超平面（切线束） |
| **对凸函数** | 始终存在 | 始终存在（至少一个） |

**例：$f(x) = |x|$ 在 $x=0$**

满足 $f(x) \geq f(0) + g \cdot (x-0)$ 即 $|x| \geq g \cdot x$ 的 $g$ 取值：

- 取 $g = 0.5$：$|x| \geq 0.5x$ ✓
- 取 $g = -0.5$：$|x| \geq -0.5x$ ✓
- 取 $g = 0$：$|x| \geq 0$ ✓
- 取 $g = 2$：当 $x < 0$ 时，$|x| = -x \geq 2x$？$x=-1$ 时 $1 \geq -2$ ✓；当 $x > 0$ 时，$x \geq 2x$ → $x \leq 0$ ✗

因此：

$$\partial f(0) = [-1, 1]$$

这是一个**闭区间**，所有在 $[-1, 1]$ 内的值都是 $f$ 在 $x=0$ 的次梯度。

**次梯度集合的通用形式**：

$$\partial f(x_0) = [\text{左导数}, \text{右导数}]$$

---

### D-4 次梯度与凸函数的必然关系

**次梯度在数学上就是为凸函数定义的概念。**

在严格的凸分析（Convex Analysis）中，次梯度的定义域被限定在凸函数上。一个函数如果不是凸函数，我们不会谈论它的次梯度——不是因为不能定义，而是因为定义本身**依赖于凸性**才能保证有意义。

**为什么？**

次梯度定义的核心是全局支撑线/超平面：$f(x) \geq f(x_0) + g^\top (x - x_0)$ 要求次梯度定义的直线必须**始终位于函数图像下方**。

凸函数的本质特征是：**任意两点之间的函数值都不高于两点连线的插值**。这个性质保证了**总存在至少一条全局支撑线**通过任意点。

非凸函数在局部可能低于任何经过该点的线性近似，因此**全局下支撑线可能不存在**。

**直观对比：**

- **凸函数** $f(x) = x^2$ 在 $x=1$ 处：梯度 $g = 2$，支撑线 $y = 1 + 2(x-1)$ 始终在 $x^2$ 下方 ✓
- **非凸函数** $f(x) = \sin(x)$ 在 $x=\pi/2$ 处：导数 $g = 0$，支撑线 $y=1$ 完全在函数图像上方（除切点外）✗

**次梯度存在的充要条件：** 对于凸函数 $f$，在**所有定义域内点上次梯度集合非空**（至少有一个次梯度）。




## 附录E：PL条件（Polyak-Łojasiewicz）详解

本附录解释1.1.2节目标函数分类中 **"满足PL条件的非凸函数 → 可证收敛到全局最优"** 的理论依据、数学含义及其在深度学习理论分析中的意义。

---



### E-1 为什么非凸函数还能保证全局收敛？

PL条件是一个比"强凸"更宽松的数学条件，但神奇的是，它仍然能保证像梯度下降法这样的基础算法，可以从**任意**起始点，以**线性速度**收敛到全局最优解。

这看似矛盾——非凸函数怎么还能保证全局收敛？关键在于PL条件的定义本身蕴含了极强的几何性质。

---

### E-2 PL条件的数学定义

PL条件的数学形式为：

$$
\frac{1}{2} \| \nabla f(\mathbf{x}) \|^2 \ge \mu \left( f(\mathbf{x}) - f(\mathbf{x}^*) \right), \quad \mu > 0
$$

其中，$f(\mathbf{x}^*)$ 代表函数的**全局最小值**，$\mu > 0$ 是一个常数。

---

### E-3 为什么PL条件能保证全局收敛？

#### E-3.1 消除"虚假"的驻点

上述不等式的直接推论是：**函数的所有临界点（即梯度为零的点）都必须是全局最优点**。

如果 $\nabla f(\mathbf{x}) = 0$，则左边为0，不等式变为 $0 \ge \mu (f(\mathbf{x}) - f(\mathbf{x}^*))$。由于 $\mu > 0$，这要求 $f(\mathbf{x}) - f(\mathbf{x}^*) \le 0$。又因为 $f(\mathbf{x}^*)$ 是全局最小值，$f(\mathbf{x}) - f(\mathbf{x}^*) \ge 0$ 恒成立。两者结合，只能有 $f(\mathbf{x}) = f(\mathbf{x}^*)$。

因此，梯度下降法不会因为陷入 **鞍点(Saddle Point)**或**局部极小点(Local Minima)** 而"迷路"。只要算法能持续沿着梯度下降的方向走，它就在稳步朝着那个唯一的"目标"——全局最优值前进。

#### E-3.2 线性收敛速率的理论保证

在满足PL条件的前提下，使用梯度下降法时，目标函数值 $f(\mathbf{x}_k)$ 的收敛速度有严格的理论保证：

$$
f(\mathbf{x}_k) - f(\mathbf{x}^*) \le \left(1 - \frac{\mu}{L}\right)^k \left( f(\mathbf{x}_0) - f(\mathbf{x}^*) \right)
$$

其中，$L$ 是梯度平滑常数（$\nabla f$ 是 $L$-Lipschitz连续的）。

这个公式说明**误差是按指数级衰减的**，即所谓的**线性收敛**——这是最理想的收敛速率之一。

---

### E-4 与强凸的关键区别

很多强凸函数（如岭回归）都满足PL条件，但**反过来不成立**。

**经典反例：欠定线性回归**

考虑函数 $f(\mathbf{x}) = \|\mathbf{A}\mathbf{x} - \mathbf{b}\|^2$，其中 $\mathbf{A}$ 的列数多于行数（欠定系统）。

- 这个函数**不是强凸的**，因为海森矩阵 $\mathbf{A}^\top\mathbf{A}$ 是**半正定**的，存在特征值为0；
- 但如果将问题限制在一个有界集上，它却**可以满足PL条件**。

这意味着，即使目标函数在某些方向上是"平坦"的（没有严格的二次弯曲），**PL条件仍然赋予了它强大的可优化性**。

---

### E-5 PL条件在深度学习理论中的意义

PL条件的强大之处在于，它为大量实际的、非凸的机器学习问题提供了理论上的收敛保证。研究表明，PL条件在许多重要场景下都被发现是成立的：

| 场景 | 说明 |
|:---|:---|
| **线性神经网络** | 在特定的初始化条件下满足PL条件 |
| **神经正切核（NTK）下的非线性网络** | 宽网络在NTK regime下满足PL条件 |
| **某些矩阵分解问题** | 如非负矩阵分解的一些变体 |
| **强化学习中的LQR问题** | 线性二次调节器满足PL条件 |

这意味着，对于这些原本复杂的非凸问题，可以从理论层面证明：简单如梯度下降法也能找到全局最优解。

---

### E-6 PL条件的延伸

PL条件的强大"基因"还被延伸到了更复杂的优化问题中：

| 延伸 | 应用场景 |
|:---|:---|
| **双边PL条件** | 非凸-非凹的极小极大优化问题（如GAN训练），可保证交替梯度下降算法找到鞍点 |
| **平均PL曲率** | 平均场神经网络的全局收敛性分析 |

---

### E-7 速查总结

| 问题 | 答案 |
|:---|:---|
| **PL条件是什么？** | Polyak-Łojasiewicz条件，比强凸更宽松但仍能保证全局线性收敛的数学条件 |
| **为什么能保证全局收敛？** | 不等式要求梯度为零的点只能是全局最优点，消除了鞍点和局部极小 |
| **与强凸的区别？** | 强凸 ⇒ PL，但PL ⇏ 强凸（PL更宽松） |
| **收敛速度如何？** | 线性收敛 $O((1-\mu/L)^k)$，与强凸相当 |
| **在深度学习中的应用？** | 为线性神经网络、NTK、矩阵分解等非凸问题提供全局收敛的理论保证 |
| **PL条件对非凸函数都成立吗？** | 不，只有满足该不等式的特定非凸函数才成立 |

---

**附录E一句话总结**：PL条件给出了一个比强凸更宽松、但依然能保证全局线性收敛的非凸函数条件。它消除了局部极小和鞍点的干扰，让梯度下降法能够从任意起点走向全局最优。在深度学习理论分析中，它为许多重要的非凸问题提供了可证明的全局收敛保证。


## 附录F：FGD与SGD的二重差异详解（完整推导与实验验证）

本附录为第3节内容的完整展开，包含详细的数学推导和实验验证。

> **符号约定**：FGD = Full‑Gradient Descent 全量梯度下降；SGD = Stochastic Gradient Descent，逐样本随机梯度下降。
> $W_s$：epoch起始参数(start)；$W_e$：epoch结束参数(end)；上标$\cdot^{FGD}$、$\cdot^{SGD}$区分两套迭代序列。

在深度学习中，我们常以为SGD和FGD的差异仅仅来自梯度噪声。严格推导可以发现，二者差异来自**两个相互独立、可以解耦的来源**：时序迭代带来的确定性差异、乱序采样带来的随机差异。

数据集 $\{X_i,y_i\},i=1\sim n$，模型 $\hat y = f(W,X)$，损失 $L(y,\hat y)$，学习率 $\eta$。

---

### F-1 第一重：时序迭代带来的确定性差异

该效应属于确定性机制，和随机无关：即使关闭shuffle，使用固定不变的样本顺序遍历，SGD与FGD的epoch输出依然会不一样。

**FGD迭代公式**

FGD在整个epoch内参数保持不变，全部样本梯度都在起始参数$W_s$处计算，一次性完成更新：
$$
W_e^{FGD}=W_s-\eta\sum_{i=1}^n \nabla L\big(y_i,f(W_s,X_i)\big)
$$
所有梯度求值点完全相同，求和满足加法交换律；**样本先后顺序不会影响FGD输出结果**。

**确定性SGD完整公式推演（无shuffle、固定顺序）**

SGD核心行为：每处理完一个样本，立刻更新参数；下一个样本的梯度，在**已经更新后的新参数**上重新计算。

设epoch起始参数为 $W_s$：

- 第1个样本，梯度在起始点$W_s$计算，得到第一步更新：
$$W_1^{SGD}=W_s-\eta \nabla L\big(y_1,f(W_s,X_1)\big)$$

- 第2个样本，梯度在更新后的 $W_1^{SGD}$ 上计算：
$$W_2^{SGD}=W_1^{SGD}-\eta \nabla L\big(y_2,f(W_1^{SGD},X_2)\big)$$

- 第$i$步通用迭代：
$$W_{i}^{SGD}=W_{i-1}^{SGD}-\eta \nabla L\big(y_i,f(W_{i-1}^{SGD},X_i)\big)$$

遍历全部$n$个样本，epoch结束得到终点参数 $W_e^{SGD}$：
$$
\boldsymbol{W_e^{SGD}=W_s - \eta \sum_{i=1}^{n} \nabla L\big(y_i,f(W_{i-1}^{SGD},X_i)\big)}
$$

**关键核心结论**

1. SGD求和中每一项梯度求值点 $W_{i-1}^{SGD}$ 互不相同，每一步梯度都依赖前面样本带来的参数改变；
2. 加法交换律失效：调换样本顺序，每一项梯度的求值点全部发生改变，最终 $W_e^{SGD}$ 必然改变；
3. 线性模型特例：当梯度与参数本身无关，该确定性差异消失，一轮SGD等价FGD。

---

### F-2 第二重：乱序（shuffle）采样带来的随机差异

随机差异产生的根源：**固定同一个模型参数$W$不变时，不同样本计算出来的样本损失不一样，对应的单样本梯度自然各不相同**。

FGD把全部样本损失求和取平均，样本个体差异互相抵消，输出唯一确定的更新，不存在该随机效应。

SGD逐样本更新，每一步只用某一个样本梯度近似全集梯度。`shuffle=True`只是对数据集做无放回随机重排，样本本身不变，只改变出场次序。

**实例看差异如何发生（$W$全程固定不动）**

数据集两个样本A、B，模型参数$W$完全固定。
- 当前参数下样本A损失大，对应单样本梯度 $g_A$幅值大；
- 当前参数下样本B损失小，对应单样本梯度 $g_B$幅值小；

1）shuffle序列[A,B]，第一步用$g_A$更新：
$$W' = W - \eta \cdot g_A$$
2）另一次shuffle得到序列[B,A]，第一步用$g_B$更新：
$$W'' = W - \eta \cdot g_B$$

$g_A \neq g_B \implies W' \neq W''$。

> **关键点**：初始输入参数完全一致；差异**不是迭代修改W造成**，仅仅shuffle随机选出不同样本；样本各自损失不同，单样本梯度不同，第一步更新量就出现差别，这就是乱序采样带来的随机差异。

该初始差别会向后传递放大：第一步得到不同参数，后续样本的损失梯度基于新参数计算，整条迭代轨迹产生随机偏移。

统计意义上，单样本梯度是全集梯度的无偏估计；但样本梯度之间存在离散，带来估计方差，也就是常说的梯度噪声。

> 📌 **和第一重确定性差异严格区分**
> - 第二重随机差异：起始$W$固定不变；根源在于样本之间损失、梯度的个体差异，shuffle随机挑选样本，造成更新量随机不同。
> - 第一重确定性差异：不需要随机；epoch内部每一步迭代直接修改$W^{SGD}$；后续损失梯度在已经改变的参数上计算。

真实训练开启shuffle时两种效应耦合在一起：shuffle引入随机梯度估计差异；紧接着每一步参数修改，触发时序迭代的确定性差异，共同决定迭代路径。

---

### F-3 四种场景，二重差异组合对照表

| 训练场景 | 第一重：时序迭代确定性差异 | 第二重：乱序采样随机差异 |
|---|---|---|
| FGD全量梯度下降 | ❌ 无 | ❌ 无 |
| 确定性SGD（无shuffle、固定顺序） | ✅ 存在 | ❌ 无（本次实验场景） |
| 真实SGD（shuffle随机乱序） | ✅ 存在 | ✅ 存在 |

---

### F-4 Mini‑batch场景下的二重差异

Mini‑batch梯度下降同时包含两套机制：

1. **第二重随机差异（batch内部）**：同一参数下，不同batch样本构成不同，子集损失分布不同，batch梯度估计结果不同；batch越小，样本越少，个体差异难以抵消，随机波动越大。

2. **第一重确定性差异（batch之间）**：每完成一个batch就更新参数，下一批样本的损失、梯度在新参数上求解，时序迭代效应持续生效。

batch增大，样本个体差异被平均，随机差异被抑制；只要batch不等于全集样本，**时序迭代带来的确定性差异始终存在**。

---

### F-5 核心常见误区

很多说法认为SGD和FGD区别只有梯度噪声。
这个认知不完整：**关闭shuffle，彻底消除随机噪声，固定顺序逐样本SGD，输出$W_e^{SGD}$依然不等于FGD输出$W_e^{FGD}$**。

SGD与FGD之间总差异 = 时序迭代带来的确定性差异 + 乱序采样带来的随机差异；二者相互独立，共同起作用。

---

### F-6 实验验证（第一重确定性差异）

**实验设定**

非线性模型 $\hat y = w^2 x$，MSE损失：
$$
L=\frac12(y-\hat y)^2
\quad\Rightarrow\quad
\frac{\partial L}{\partial w}=-(y-w^2 x)\cdot 2wx
$$

样本：$point_1=(x=1.0,y=2.0),\ point_2=(x=2.0,y=3.0)$
- epoch起始参数 $w_s=1.0$
- 学习率 $\eta=0.05$
- 使用两套固定顺序，关闭随机，对比epoch终点输出 $w_e^{SGD}$。

**实验代码见附录F末尾的代码单元**。

**实验结果解读**

1. 无任何随机，仅仅调换样本顺序，$w_e^{SGD}$就发生变化，验证第一重**时序迭代确定性差异真实存在**；
2. 两套SGD终点，均不等于FGD输出 $w_e^{FGD}$；即使消除全部随机噪声，SGD与FGD输出依然不一致。

**真实训练总结**

开启shuffle的真实SGD，两套效应同时起作用：
- 时序迭代确定性差异：epoch内部参数不断被改写，样本顺序改变迭代路径；
- 乱序采样随机差异：样本损失梯度本身离散，shuffle引入梯度估计噪声，帮助逃离局部极小，提升泛化。

> **注意**：梯度噪声是**盲目扰动，属于副作用**，没有理论保证一定逃离局部极小；只是存在概率跳出部分局部极小，既不能保证一定跳出，也不能保证跳向更优的参数区域。实践中观察到泛化能力提升是统计层面的经验现象。


```python
import numpy as np

def grad_L(w, x, y):
    """
    计算单样本损失对参数w的梯度
    模型：yhat = w² * x
    损失：MSE  L = 1/2*(y‑yhat)²
    解析求导：dL/dw = -(y − w²·x) · 2·w·x
    """
    yhat = (w ** 2) * x
    grad = -(y - yhat) * (2 * w * x)
    return grad


def deterministic_sgd_epoch(w_s, eta, data_order):
    """
    【确定性SGD】完整遍历一轮epoch，无shuffle，样本顺序固定
    核心行为：每处理1个样本，立刻更新参数；下一个样本梯度基于更新后的w计算。
    """
    w = w_s
    trace = [w]
    for (x, y) in data_order:
        g = grad_L(w, x, y)
        w = w - eta * g
        trace.append(w)
    return w, trace


def full_gd_step(w_s, eta, data):
    """
    FGD全量梯度下降：一个epoch只执行1次参数更新
    全部n个样本梯度统一在**同一个起始参数w_s**处计算，求和之后一次性更新。
    """
    total_grad = 0.0
    for (x, y) in data:
        total_grad += grad_L(w_s, x, y)
    w_fgd = w_s - eta * total_grad
    return w_fgd


# ----------------------运行实验-------------------------
point1 = (1.0, 2.0)
point2 = (2.0, 3.0)

w_s = 1.0
eta = 0.05

# 顺序A：先point1，后point2
w_e_A, trace_A = deterministic_sgd_epoch(w_s, eta, [point1, point2])
# 顺序B：先point2，后point1
w_e_B, trace_B = deterministic_sgd_epoch(w_s, eta, [point2, point1])
# FGD全量梯度下降作为参照基准
w_fgd_out = full_gd_step(w_s, eta, [point1, point2])

print(f"epoch起始参数 w_s = {w_s:.6f}")
print(f"\n顺序 [point1, point2]，迭代轨迹：{[round(v, 6) for v in trace_A]}")
print(f"epoch终点 w_e^SGD(A) = {w_e_A:.6f}")
print(f"\n顺序 [point2, point1]，迭代轨迹：{[round(v, 6) for v in trace_B]}")
print(f"epoch终点 w_e^SGD(B) = {w_e_B:.6f}")
print(f"\nFGD输出 w_e^FGD = {w_fgd_out:.6f}")
print(f"\nw_e_A != w_e_B ? {not np.isclose(w_e_A, w_e_B)}")


## 附录G：步长敏感性的理论根基



### G-1 问题的核心

1.2.1节使用了一维凸二次函数 $f(x)=0.5x^2-2x$ 作为分析对象，推导出误差递推式 $e_k=(1-\eta)^k e_0$，并以此说明步长选择对收敛行为的影响。

读者可能会问：**这只是一个特例，凭什么说步长敏感是梯度下降法的普遍缺陷？**

本附录从三个层次回答这个问题，由浅入深：

1. **泰勒展开根源**：一阶近似的普适误差结构
2. **Lipschitz连续性定理**：所有光滑函数的收敛临界值
3. **病态曲率的高维灾难**：特例掩盖的真实矛盾

---

### G-2 第一层：泰勒展开——普适的误差根源

无论目标函数 $f(\theta)$ 的具体形式是什么（只要它二阶可导），梯度下降法的每一步更新 $\theta_{k+1} = \theta_k - \eta \nabla f(\theta_k)$ 都可以用**泰勒展开**来分析其误差结构。

对 $f$ 在 $\theta_k$ 处展开到二阶：

$$
f(\theta_{k+1}) = f(\theta_k) + \nabla f(\theta_k)^\top (\theta_{k+1} - \theta_k) + \frac{1}{2} (\theta_{k+1} - \theta_k)^\top \nabla^2 f(\xi_k) (\theta_{k+1} - \theta_k)
$$

其中 $\xi_k$ 位于 $\theta_k$ 和 $\theta_{k+1}$ 之间，$\nabla^2 f$ 是海森矩阵（Hessian）。

代入更新公式 $\theta_{k+1} - \theta_k = -\eta \nabla f(\theta_k)$：

$$
f(\theta_{k+1}) = f(\theta_k) - \eta \|\nabla f(\theta_k)\|^2 + \frac{\eta^2}{2} \nabla f(\theta_k)^\top \nabla^2 f(\xi_k) \nabla f(\theta_k)
$$

**关键洞察**：

- 当 $\eta \to 0$ 时，二阶项 $O(\eta^2)$ 可忽略，函数值**必定下降**；
- 当 $\eta$ 增大时，二阶项的符号和大小由**海森矩阵的曲率**决定；
- 一旦 $\eta$ 超过某个临界值，二阶项（可能为负，即函数值上升）的绝对值超过线性项，下降条件 $f(\theta_{k+1}) < f(\theta_k)$ 被打破。

**这个结论与函数的具体形式无关**——只要是二阶可导的函数，就必然存在这样的临界步长。凸二次函数的特例只是让这个临界值可以被解析地计算出来（$\eta_{\text{crit}}=2$），但其背后的数学机制是普适的。

---

### G-3 第二层：Lipschitz连续性——普适的收敛临界定理

在优化理论中，存在一个针对**所有光滑函数**的普适定理。

**定义**：若目标函数的梯度满足 **$L$-Lipschitz 连续**，即：

$$
\|\nabla f(\theta_1) - \nabla f(\theta_2)\| \leq L \|\theta_1 - \theta_2\|, \quad \forall \theta_1, \theta_2
$$

则梯度下降法收敛的**充分条件**为：

$$
\boxed{\eta \leq \frac{2}{L}}
$$

**直观理解**：

- 常数 $L$ 是梯度的**最大变化率**，本质上反映了损失地形的**最陡峭程度**；
- 1.2.1节的二次函数 $f(x)=0.5x^2-2x$，其二阶导数（曲率）恒为 1，因此 $L=1$，临界值 $2/L=2$；
- 对于任意函数，$L$ 可能远大于1（地形更陡峭），临界步长可能远小于2。

**关键结论**：

虽然不同函数的具体 $L$ 值不同，最优步长也因此不同，但 **"步长超过 $2/L$ 就必然导致发散"** 这一现象对所有满足Lipschitz连续条件的函数都是普适的。

几乎所有深度学习的损失函数都满足局部Lipschitz条件（在参数的紧致子集上），因此这个临界约束在深度学习中具有普适性。

---

### G-4 第三层：病态曲率——特例掩盖的真实困境

你可能想进一步追问："既然存在临界值 $2/L$，那我把步长设得很小不就行了？"

这正是1.2.1节特例中**没有暴露、但在高维深度学习中极其普遍**的更深刻矛盾——**病态曲率（Ill-conditioning）**。

在深度神经网络的参数空间中，不同方向的曲率（即海森矩阵的特征值）差异可达数个数量级：

| 方向类型 | 曲率大小（$\lambda$） | 对应的 $L$ 值 | 对步长的要求 |
|:---|:---|:---|:---|
| **陡峭方向** | $\lambda_{\max}$ 极大（如 $10^4$） | $L \geq \lambda_{\max}$ | 必须极小步长，否则震荡发散 |
| **平缓方向** | $\lambda_{\min}$ 极小（如 $10^{-4}$） | $L \geq \lambda_{\min}$ | 需要极大步长，否则移动极其缓慢 |

标准GD给**所有方向施加同一个全局步长 $\eta$**：

- 你必须把 $\eta$ 设得低于陡峭方向的稳定上限 $2/\lambda_{\max}$，否则在该方向震荡发散；
- 但在平缓方向，这个被迫缩小的步长远小于其需求 $2/\lambda_{\min}$，收敛极其缓慢。

这就是**条件数（Condition Number）** 困境：

$$
\boxed{\kappa = \frac{\lambda_{\max}}{\lambda_{\min}} \gg 1 \quad \Rightarrow \quad \text{统一步长 }\eta\text{ 永远无法同时满足所有方向}}
$$

1.2.1节的二次函数由于 $\lambda_{\max} = \lambda_{\min}$（各向同性），$\kappa=1$，完美地掩盖了这种矛盾。而在真实的深度学习中，$\kappa$ 可能高达 $10^3$ 到 $10^6$。

**这正是后续优化算法（AdaGrad的逐参数步长、RMSprop的自适应缩放、Momentum的方向平滑）诞生的根本工程动机**——它们分别从不同角度回应这个"统一步长无法适配所有方向"的根本矛盾。

---

### G-5 三层次总结

| 层次 | 核心论证 | 普适性范围 | 1.2.1特例的角色 |
|:---|:---|:---|:---|
| **① 泰勒展开根源** | $f(\theta_{k+1}) = f(\theta_k) - \eta\|\nabla f\|^2 + O(\eta^2)$，有限步长下二阶项必然介入 | 所有二阶可导函数 | 演示了二阶项如何导致发散 |
| **② Lipschitz定理** | 收敛必要条件 $\eta \leq 2/L$，$L$ 为梯度最大变化率 | 所有梯度Lipschitz连续的函数 | $L=1$ 时临界值为2，可解析验证 |
| **③ 病态曲率** | 不同方向 $\lambda$ 差异巨大，统一步长无法适配 | 所有高维非凸问题 | $\kappa=1$ 各向同性，**掩盖了真实矛盾** |

**结论**：1.2.1节虽然使用了**凸二次函数特例**进行推导，但它揭示的是**一阶优化算法在面对二阶曲率信息时的内在脆弱性**——从泰勒展开的普适误差结构，到Lipschitz临界值的普适约束，再到病态曲率下统一步长的根本性失效，三个层次层层递进，共同说明"步长敏感"是梯度下降法无法回避的普遍性缺陷。这种脆弱性在结构更复杂的非凸、高维函数中**只会加剧，而不会消失**。

---

**附录G一句话总结**：一维凸二次函数特例的价值在于它提供了可解析验证的教学模板，但"步长敏感"的普遍性根源在于泰勒展开的普适误差结构、Lipschitz临界值的普适约束以及病态曲率下统一步长的根本性失效——这三个机制共同保证了该缺陷在所有光滑函数上的普遍存在。

